# 2.1 LSTM with target-aware attention — Stage 1 relevance and catalogue shortlist

**Experiment ID:** `2.1_lstm` · **Track:** neural networks · **Stage 1:** learn
personal click relevance from recorded shown/clicked pairs, then score every
article eligible at each request's timestamp and export a top-100 shortlist for
Stage 2 (business reranking, implemented in the shared stage).

**Quick start (Colab, GPU runtime):** the default `RUN_SCOPE = "smoke"` is the
recommended first execution:
- 500 hash-sampled training requests and one epoch.
- Checkpoint selection on 20 validation requests.
- Catalogue export of 5 requests per split.
- No controls and no Optuna; the resume check is on.

`RUN_SCOPE = "aligned_dev"` uses 3.1's development limits: 5,000 training
requests, 100 validation-selection requests and 25 export requests per split.

In every scope, requests are sampled with the shared hash rule from the
**complete** population, and each selected request is scored against its **full**
eligible union-catalogue pool. Sampled outputs are labelled `sampled_requests`.

It trains, exports the recorded-slate diagnostics and the sampled catalogue
shortlists, persists the run to Drive, and reloads it from the Drive copy.
**Full scoring is a separate, deliberate configuration:** set `RUN_SCOPE = "full"`. Section 7
estimates catalogue pairs, export size and runtime before it scores anything. To
export from a saved model without training, set `TRAIN_MODEL = False` and
`LOAD_RUN_ID`.

Code lives next to this notebook in `lstm_data.py` (data, metrics, exports; no
PyTorch) and `lstm_model.py` (network, batching, training). Both are copied into
every saved model bundle, so the bundle can be reloaded without this repository.

## 1. Business hypothesis and experiment overview

| Item | Detail |
| --- | --- |
| ID / owner | `2.1_lstm` / see `OWNER` in Section 2 |
| Model | Chronological history → LSTM → candidate-as-query masked attention → MLP head (PyTorch) |
| Business objective | Interim report §1.1 and §1.2.3: relevant recommendations that support continued reading and exploration; this model supplies the **initial relevance score** for every candidate, and the shared stage reranks for continuity, freshness and diversity |
| Training pool | Recorded `article_ids_inview` slates (clicked = 1, shown but unclicked = 0); no catalogue negatives |
| Prediction pool | Every catalogue article eligible at the request time (`time_eligible_catalogue`); unshown outcomes are **unknown** |
| Outputs | Top-100 shortlist per request; catalogue scores (`score` = BCE logit, `click_probability` = uncalibrated sigmoid); separate recorded-slate diagnostic scores |
| Metrics | **Primary (selection):** catalogue known-click Recall@100 on validation, full eligible pool per request. **Secondary / tie-break:** observed-click nDCG@100. **Diagnostic:** recorded-slate impression AUC. nDCG@5 belongs to Stage 2's final list |

**Hypothesis.** A reader's interests are not one static profile. Within a news
history, recent reads, repeat visits and reading depth say which interest is
active right now, and different candidates relate to different parts of that
history. So:

1. **Target-aware attention.** Weighting history by relevance *to each candidate*
   (a football article attends to past sport reads, a crime story to past crime
   reads) should rank candidates better than one fixed summary of the user.
2. **Order.** Reading the history in chronological order with an LSTM should beat
   the same events in a random order, if the sequence carries drift and session
   information beyond each event's own timestamp.
3. **Engagement and content.** Read time and scroll depth from past reads, and
   frozen text and image vectors, should let the model score articles it has
   never seen, which a news catalogue produces constantly.

**Focused validation checks** (development data, same candidate rows):

| Comparison | Variant | Evidence that supports keeping the more complex choice |
| --- | --- | --- |
| Target-aware attention vs simpler summary | `attention` vs `last_state` (final LSTM state, same for all candidates) | Paired validation Recall@100 gain (same requests, full pools) whose user-cluster bootstrap 95% CI excludes 0, plus attention concentrated on related history (category-match lift > 1); AUC shown as a diagnostic |
| Ordered vs shuffled history (order only) | `attention` vs `attention_shuffled`: identical per-event inputs, computed in true chronological order (article, read time, scroll, age at read, gap to the previous event, timestamp and so the gap to the impression), fed to the LSTM in a fixed random order | Same rule (Recall@100). If the CI includes 0, the LSTM's use of sequence order adds nothing measurable beyond the per-event time features, and a cheaper pooled model may be enough |

A third control, `attention_no_lstm` (attention over per-event inputs with no
recurrence), is available but off by default.

**What this evidence cannot show.** These are offline click-ranking results on
logged slates. A gain supports a *relevance* claim only. It does not show more
continued reading or retention, which need session-continuation outcomes or
online tests (shared evaluation stage). Attention weights are diagnostics of what
the model uses, not causal explanations of reader behaviour.

**Leakage controls.** Inputs are limited to information available at impression
time:
- Histories come from each split's fixed pre-period history file. Every event is
  asserted to precede the split's first impression. Later in-period clicks are not
  appended.
- Current or next read time and scroll, `total_inviews`, `total_pageviews` and
  `total_read_time` are never read.
- The official test split has no labels and is never used for any decision.

### Design decisions and what would support them

| Decision | Starting choice | Rationale / evidence to check after the run |
| --- | --- | --- |
| History source | Raw `history.parquet` lists (article, time, read time, scroll) per split; the engineered table has only aggregates | Preflight shows aligned lists, chronological sort, cutoff respected |
| History length | Most recent **50** events (configurable) | Covers the median EB-NeRD history (report §2.1.2: median 80–92 on LARGE) while bounding cost. Not tuned; Section 6 reports Recall@100 by history length |
| Empty history | Learned "empty history" context vector + `history_empty` flag | Rare in this data. Counts reported in Section 3 |
| Unknown / missing articles | Kept; category/type "missing"/"unknown" indices; zero vector **plus** missing flags for absent text/image vectors | Coverage table in Section 4 |
| Article representation | Category, type, topics (shared train vocabulary), premium, body length, sentiment, frozen Gemini text (768-d) and EB-NeRD image (1024-d) vectors, projected **separately** | Scores unseen articles. Learned article-ID embedding is optional (off), so IDs are never the only route |
| Recency | Log hours from each history event to the impression as an attention bias; log age of each article at read time and inter-event gap as LSTM inputs | DeNA/Tom3TK found time features strongest among leakage-free signals |
| Candidate freshness | Log article age at the request, computed from the catalogue (the same rule as the shared `article_age_h`, checked against it in Section 4) | Constructible for any catalogue article. The earlier within-slate age rank and recorded slate size were **removed**, because a catalogue request has no recorded slate |
| Catalogue pool | `published_at_or_before_request_v1` (shared with 3.1): the ordered union train_val → test catalogue (first record per ID) for **every** split; known publication time ≤ request time; no age/access cutoff; no embedding filter | One pool across models enables comparison on identical request IDs. Section 4 reports pair counts and coverage; Section 7 the union-wide costs |
| Articles without vectors | Kept, with zero placeholder + `has_text`/`has_image` flags. Training hides the text vector of 20% of candidates (`text_dropout`) so text-less scoring is learned | Most test-catalogue articles have no text vector. Section 4 reports pair-weighted coverage; Section 7 reports the text-vector share inside shortlists |
| Objective | BCE on logits over every candidate (multiple clicks allowed); not the course's SmoothL1 rating loss | Click target is binary |
| Optimiser | AdamW (lr 1e-3, wd 1e-5), grad-clip 1.0, ReduceLROnPlateau on the selection objective, early stopping (patience 2 selection evaluations), AMP on GPU | Standard, course-connected. AMP only speeds up training |
| Selection | Once per epoch (the declared interval): exact top-100 over the **full** eligible validation catalogue for a fixed, seeded sample of validation requests; maximise macro known-click Recall@100, then observed-click nDCG@100, then the earlier checkpoint. BCE still trains on observed pairs | Contract (Experiments README, validation protocol). Stage 2 cannot recover a click Stage 1 discards. Validation is **development data**; the selected model is then scored on the full validation catalogue in Section 7 |

## 2. Setup and run configuration

In [ ]:
# ---------------------------------------------------------------- run configuration
# Edit this cell only. Recommended first execution: the defaults below (smoke run,
# no ablations, no Optuna, resume check on). Then set RUN_SCOPE (below) for the full
# main model, and run the controls in a separate session (see Section 5).
from pathlib import Path

EXPERIMENT_ID = "2.1_lstm"
OWNER = "Winston Ho Rui Wei"          # edit if someone else runs this
SEED = 42
FEATURE_CASE = "raw"   # only keys, labels and a few context columns are read; see Section 3

# Code version: the helper modules are checked out at this reviewed commit and their
# sha256 must match. Colab uses your existing GitHub credentials for the clone/fetch.
REPO_URL = "https://github.com/KoalaLyfestyle/BT4222-Final-Project.git"
REPO_BRANCH = "claude/serene-hawking-gkuqc7"
CODE_COMMIT = "52deb2fd89c32b43be99bdd62d3bf812c6487228"
EXPECTED_MODULE_SHA256 = {
    "lstm_data.py": "2faf7594bb53d0b0ffe0415c682a058dcdbe4f2101dd92ea30f8afc6ef62dc54",
    "lstm_model.py": "8adef32ebd0b72c9189006765853a38c6c8d65de1e30acfa9b972a84e3a63890"
}
CLONE_DIR = Path("/content/BT4222-Final-Project")

# Inputs (read-only). The shared folder is owned by another account: add a Drive
# shortcut ("Add shortcut to Drive") so it appears under MyDrive.
DRIVE_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder"),
    Path("/content/drive/MyDrive/BT4222 Shared Final Folder"),
]
DATA_ROOT = None        # set explicitly to override, e.g. a mini sample folder
PATH_OVERRIDES = {}     # {key: path} for any input in lstm_data.PATH_CANDIDATES

# Results are saved in the shared project folder under
# DATA_ROOT/Experiments/2.1_lstm/<run_id>/ (resolved after finding DATA_ROOT).

# Train / load switches
TRAIN_MODEL = True       # False: load LOAD_RUN_ID from the output root and export
LOAD_RUN_ID = None       # e.g. "20261104_0930_seed42"
RESUME_RUN_ID = None     # continue an interrupted run folder (training, controls or
                         # catalogue splits); with TRAIN_MODEL=False also set LOAD_RUN_ID

# Request scope (aligned with 3.1_pretrained_similarity). Requests are sampled, never
# articles: every selected request is scored against its FULL eligible pool.
#   "smoke"       - tiny labelled smoke run
#   "aligned_dev" - 3.1's development limits (5,000 train / 100 selection / 25 export)
#   "full"        - every training and export request (deliberate, large)
RUN_SCOPE = "smoke"
SCOPE_LIMITS = {
    "smoke":       {"train_requests": 500,  "selection_requests": 20,  "export_requests": 5},
    "aligned_dev": {"train_requests": 5000, "selection_requests": 100, "export_requests": 25},
    "full":        {"train_requests": None, "selection_requests": 100, "export_requests": None},
}
SAMPLING_SEED = 42       # shared request-sampling seed (BLAKE2b of "seed:split:behaviour_id")
# Optional: 3.1's saved request IDs, to reuse/compare (paths under Drive; None = skip)
REFERENCE_INPUT_AUDIT = None        # e.g. ".../3.1_pretrained_similarity/<run>/input_audit.json"
REFERENCE_TRAINING_IDS = None       # e.g. ".../training_request_ids.json"
CHECK_AGAINST_BEHAVIOURS = True   # confirm feature rows == exploded in-view slates
HASH_INPUTS = True       # sha256 of every input (a few minutes on Drive)
RUN_RESUME_CHECK = True  # short interrupted-run/resume check (Section 5), separate folder
WRITE_COMPATIBLE_VECTORS = False  # optional: write verified list<float> copies of the vectors

# History
HISTORY_MAX_LEN = 50

# Model (starting values; defensible, not claimed optimal)
MODEL_VARIANT = "attention"
MODEL_HPARAMS = dict(
    article_dim=128, category_dim=16, type_dim=8, topic_dim=16, text_dim=64, image_dim=32,
    lstm_hidden=128, lstm_layers=1, attention_dim=64, interaction_dim=64,
    head_hidden=(256, 128), dropout=0.2, embedding_dropout=0.1,
    use_id_embedding=False, id_dim=16,
    text_dropout=0.2,   # training only: hide 20% of candidate text vectors (see Section 1)
)
ID_EMBEDDING_MIN_COUNT = 5   # only used when use_id_embedding=True
TRAIN_HPARAMS = dict(
    max_epochs=8, batch_impressions=256, learning_rate=1e-3, weight_decay=1e-5,
    grad_clip=1.0, early_stopping_patience=2, lr_factor=0.5, lr_patience=0,
    min_lr=1e-5, history_dropout=0.0, use_amp=True, eval_batch_impressions=256,
    max_train_impressions_per_epoch=None,
)

# Comparisons (Section 6): same data, hyperparameters and budget as the main model.
# Off for the first execution; later run them with TRAIN_MODEL=False + LOAD_RUN_ID
# (main model loaded, controls trained) or alongside a full training run.
RUN_ABLATIONS = False
ABLATION_VARIANTS = ["last_state", "attention_shuffled"]   # + "attention_no_lstm"
SHUFFLE_SEED = 2024
BOOTSTRAP_SAMPLES = 1000

# Optional Optuna search (development validation; never test)
RUN_OPTUNA = False
APPLY_BEST_TRIAL = False      # copy the best trial's values into the configs above
OPTUNA_STUDY_NAME = "lstm_attention_recall100_v1"   # Recall@100 objective (new study)
OPTUNA_N_TRIALS = 12
OPTUNA_TIMEOUT_MIN = 150
OPTUNA_EPOCHS = 3
OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH = 60_000

# Model selection (declared before training; see Section 5)
SELECTION_OBJECTIVE = "known_click_recall@100"     # primary, maximised
SELECTION_TIEBREAK = "observed_click_ndcg@100"     # then this, then the earlier checkpoint
SELECTION_INTERVAL = "every epoch"

# Catalogue recommendation (Stage 1 shortlist; see Section 7). Pool policy:
# published_at_or_before_request_v1 - ordered union train_val then test catalogue,
# first record per article ID, the SAME union for every split, known
# published_time <= request time, no age/access cutoff, no embedding filter.
SHORTLIST_SIZE = 100
FINAL_DISPLAY_SIZE = 5              # Stage 2's output size (recorded only)
MAX_ARTICLE_AGE_HOURS = None        # fixed by the shared policy (no age cutoff); do not change
CATALOGUE_SPLITS = ["train", "validation", "test"]
FULL_SCORE_EXPORT = "auto"          # "auto": only if it fits the limit below; True; False
FULL_SCORE_EXPORT_MAX_GB = 5.0      # per split
SAMPLED_SCORE_REQUESTS = 200        # per split, full pools, when full scores are not exported
CATALOGUE_CHUNK = 32768             # articles scored per GPU step
MAX_CATALOGUE_HOURS = 6.0           # stop before scoring if the timed estimate exceeds this
CONSISTENCY_CHECK_REQUESTS = 50     # catalogue path vs recorded-slate path, per split
CATALOGUE_RELOAD_REQUESTS = 20      # per split, Section 8

# Exports and persistence
EXPORT_CLICK_PROBABILITY = True
PERSIST_TO_DRIVE = True
PERSIST_DURING_TRAINING = True
RELOAD_CHECK_IMPRESSIONS = 300
RELOAD_TOLERANCE = 1e-4       # max |logit difference| (fp32 GPU/CPU nondeterminism)

if RUN_SCOPE not in SCOPE_LIMITS:
    raise ValueError(f"RUN_SCOPE must be one of {list(SCOPE_LIMITS)}")
TRAIN_REQUESTS = SCOPE_LIMITS[RUN_SCOPE]["train_requests"]
SELECTION_REQUESTS = SCOPE_LIMITS[RUN_SCOPE]["selection_requests"]
EXPORT_REQUESTS = SCOPE_LIMITS[RUN_SCOPE]["export_requests"]
SMOKE_TEST = RUN_SCOPE == "smoke"
if SMOKE_TEST:
    TRAIN_HPARAMS.update(max_epochs=1, batch_impressions=64)
    OPTUNA_N_TRIALS, OPTUNA_EPOCHS = 2, 1

In [ ]:
# Environment: mount Drive, check out the pinned code commit, verify module hashes.
import os, sys, subprocess, platform, json, shutil, time, math, hashlib, importlib, importlib.util
try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def git(*args, cwd=None):
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        # stderr can echo the remote URL; hide any embedded credentials.
        message = result.stderr.replace(REPO_URL, "<REPO_URL>")
        raise RuntimeError(f"git {args[0]} failed: {message.strip()[:500]}")
    return result.stdout.strip()

def has_commit(repo, commit):
    return subprocess.run(["git", "cat-file", "-e", f"{commit}^{{commit}}"], cwd=repo,
                          capture_output=True).returncode == 0

uploaded_dir = next((p for p in (Path.cwd(), Path("/content"))
                     if all((p / name).is_file() for name in EXPECTED_MODULE_SHA256)), None)
using_uploaded_helpers = IN_COLAB and uploaded_dir is not None
if using_uploaded_helpers:
    CODE_DIR = uploaded_dir
elif IN_COLAB:
    if not (CLONE_DIR / ".git").is_dir():
        git("clone", REPO_URL, str(CLONE_DIR))   # default branch; the pin is fetched below
    elif git("status", "--porcelain", cwd=CLONE_DIR):
        raise RuntimeError(f"{CLONE_DIR} has local changes; commit/stash them or delete the "
                           "folder so the pinned commit can be checked out cleanly.")
    if not has_commit(CLONE_DIR, CODE_COMMIT):
        try:
            git("fetch", "origin", REPO_BRANCH, cwd=CLONE_DIR)
        except RuntimeError as error:   # e.g. branch deleted after merge
            print("Branch fetch failed; fetching the pinned commit directly:", error)
    if not has_commit(CLONE_DIR, CODE_COMMIT):
        git("fetch", "origin", CODE_COMMIT, cwd=CLONE_DIR)
    git("checkout", "--quiet", "--detach", CODE_COMMIT, cwd=CLONE_DIR)
    CODE_DIR = CLONE_DIR / "Experiments" / EXPERIMENT_ID
else:   # local development run from a repository checkout
    CODE_DIR = next(p for p in (Path.cwd(), Path.cwd() / "Experiments" / EXPERIMENT_ID)
                    if (p / "lstm_data.py").is_file())

actual = {name: hashlib.sha256((CODE_DIR / name).read_bytes()).hexdigest()
          for name in EXPECTED_MODULE_SHA256}
mismatch = {n: actual[n] for n in actual if actual[n] != EXPECTED_MODULE_SHA256[n]}
if mismatch:
    raise RuntimeError(f"Helper modules in {CODE_DIR} differ from the reviewed versions "
                       f"pinned in this notebook: {sorted(mismatch)}")
HEAD = None if using_uploaded_helpers else git("rev-parse", "HEAD", cwd=CODE_DIR)
if IN_COLAB and not using_uploaded_helpers and HEAD != CODE_COMMIT:
    raise RuntimeError(f"checked-out commit {HEAD} != pinned {CODE_COMMIT}")
GIT_INFO = {"commit": HEAD, "pinned_commit": CODE_COMMIT,
            "branch": None if using_uploaded_helpers else REPO_BRANCH,
            "uncommitted_changes": (None if using_uploaded_helpers else
                                    bool(git("status", "--porcelain", cwd=CODE_DIR))),
            "source": "uploaded_helpers" if using_uploaded_helpers else "git_checkout",
            "module_sha256": actual, "code_dir": str(CODE_DIR)}
print("Code:", json.dumps(GIT_INFO, indent=1))

In [ ]:
# Dependencies: check before importing the helpers (DuckDB is the fallback vector reader).
from importlib import metadata

def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None

def version_tuple(text):
    return tuple(int("".join(ch for ch in part if ch.isdigit()) or 0)
                 for part in text.split("+")[0].split(".")[:3])

REQUIRED = {"torch": "2.1", "numpy": "1.24", "pandas": "2.0", "pyarrow": "14.0",
            "matplotlib": "3.5", "duckdb": "1.0"}
if RUN_OPTUNA:
    REQUIRED["optuna"] = "3.4"
INSTALL_IF_MISSING = {"duckdb", "optuna"}   # never upgrade Colab's core stack in place
rows = []
for dist, minimum in REQUIRED.items():
    version = installed(dist)
    if version is None and dist in INSTALL_IF_MISSING:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{dist}>={minimum}"],
                       check=True)
        version = installed(dist)
    ok = version is not None and version_tuple(version) >= version_tuple(minimum)
    rows.append({"package": dist, "installed": version, "minimum": minimum, "ok": ok})
print("\n".join(f"{r['package']:11s} {str(r['installed']):14s} >= {r['minimum']:6s} "
                f"{'ok' if r['ok'] else 'MISSING/OLD'}" for r in rows))
bad = [r["package"] for r in rows if not r["ok"]]
if bad:
    raise RuntimeError(f"Missing or too old: {bad}. Install them, restart the runtime, rerun.")

for name in ("lstm_data", "lstm_model"):        # drop stale copies from earlier runs
    sys.modules.pop(name, None)
sys.path = [p for p in sys.path if not p.endswith("2.1_lstm")]
sys.path.insert(0, str(CODE_DIR))
import numpy as np
import pandas as pd
import pyarrow as pa
import torch
import lstm_data as D
import lstm_model as M
assert Path(D.__file__).resolve().parent == CODE_DIR.resolve()
assert Path(M.__file__).resolve().parent == CODE_DIR.resolve()
print("Helpers imported from", CODE_DIR)

In [ ]:
# Inputs, output folders (with a write test), run ID, seed and device.
if DATA_ROOT is None:
    DATA_ROOT = next((p for p in DRIVE_ROOT_CANDIDATES if p.is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"No shared data folder found in {DRIVE_ROOT_CANDIDATES}. "
                            "Add a Drive shortcut or set DATA_ROOT.")
DATA_ROOT = Path(DATA_ROOT)
OUTPUT_ROOT = DATA_ROOT
PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT, PATH_OVERRIDES)
for key, path in PATHS.items():
    print(f"{key:30s} {path}")
for note in PATH_NOTES:
    print("NOTE:", note)
IS_MINI_SAMPLE = (DATA_ROOT / "sample_manifest.json").is_file()
if IS_MINI_SAMPLE:
    print("WARNING: DATA_ROOT is the mini implementation sample; results are not benchmarks.")

def check_writable(folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    probe = folder / f".write_probe_{os.getpid()}_{time.time_ns()}"
    payload = os.urandom(4096)
    with open(probe, "wb") as handle:
        handle.write(payload); handle.flush(); os.fsync(handle.fileno())
    try:
        if probe.read_bytes() != payload:
            raise OSError(f"read-back mismatch in {folder}")
    finally:
        probe.unlink(missing_ok=True)
    return str(folder)

PERSIST_ROOT = Path(OUTPUT_ROOT) / "Experiments" / EXPERIMENT_ID
LOCAL_ROOT = (Path("/content/bt4222_outputs") / EXPERIMENT_ID if IN_COLAB
              else CODE_DIR / "outputs")
print("Local output root writable:", check_writable(LOCAL_ROOT))
if PERSIST_TO_DRIVE:
    print("Persistent output root writable:", check_writable(PERSIST_ROOT))

stamp = time.strftime("%Y%m%d_%H%M")
if not TRAIN_MODEL:
    # Saved settings that shape the inputs (and any controls trained now) must be
    # known before Section 3.
    if not LOAD_RUN_ID:
        raise ValueError("TRAIN_MODEL=False needs LOAD_RUN_ID")
    LOAD_DIR = PERSIST_ROOT / str(LOAD_RUN_ID) / "model"
    if not LOAD_DIR.is_dir():
        raise FileNotFoundError(f"Saved model bundle not found: {LOAD_DIR}")
    saved_config = json.loads((LOAD_DIR / "model_config.json").read_text())
    HISTORY_MAX_LEN = json.loads((LOAD_DIR / "preprocessing.json").read_text())["history_max_len"]
    MODEL_VARIANT = saved_config["variant"]
    MODEL_HPARAMS = {k: saved_config[k] for k in MODEL_HPARAMS}
    source_config = json.loads((PERSIST_ROOT / LOAD_RUN_ID / "run_config.json").read_text())
    TRAIN_HPARAMS = dict(source_config["training"]["config"])
    SEED = source_config["seed"]
    SOURCE_SELECTION = source_config.get("selection")
    SOURCE_PROTOCOL = source_config.get("protocol") or {}
    # Provenance of the WEIGHTS (follows load-of-load chains back to the training run).
    WEIGHTS_SELECTED_UNDER = source_config.get("weights_selected_under") or {
        "protocol_version": SOURCE_PROTOCOL.get("version", "legacy"),
        "protocol_sha256": (SOURCE_SELECTION or {}).get("protocol_sha256"),
        "objective": (SOURCE_SELECTION or {}).get("objective", "unknown (legacy AUC selection)"),
        "trained_in_run": LOAD_RUN_ID}
    SAME_PROTOCOL = (WEIGHTS_SELECTED_UNDER["protocol_version"] == "lstm_catalogue_protocol_v2"
                     and SOURCE_PROTOCOL.get("version") == "lstm_catalogue_protocol_v2")
    if SAME_PROTOCOL:
        # Same protocol: reproduce the source's scope so controls and evaluation use
        # exactly the same requests and pools (checked against its hashes in Section 4).
        RUN_SCOPE = source_config["request_scope"]["run_scope"]
        SAMPLING_SEED = source_config["request_scope"]["sampling_seed"]
        TRAIN_REQUESTS = source_config["request_scope"]["train_requests"]
        SELECTION_REQUESTS = source_config["request_scope"]["selection_requests"]
        EXPORT_REQUESTS = source_config["request_scope"]["export_requests"]
        SMOKE_TEST = RUN_SCOPE == "smoke"
        SELECTION_OBJECTIVE = SOURCE_SELECTION["objective"]
        SELECTION_TIEBREAK = SOURCE_SELECTION["tiebreak"]
    else:
        # Older pool/sample protocol: the weights may be reused for a NEW evaluation/
        # export under the current protocol, keeping the original selection provenance.
        if RUN_ABLATIONS:
            raise RuntimeError("Controls must share the main model's protocol; retrain the main "
                               "model under the current protocol first.")
        print("NOTE: the source run used an earlier pool/sample protocol. Its weights are "
              "reused for a new evaluation/export under the current protocol; its original "
              "selection provenance is kept and it is not described as selected here.")
    print(f"Load-only: saved HISTORY_MAX_LEN={HISTORY_MAX_LEN}, variant={MODEL_VARIANT}, "
          f"seed={SEED}, scope={RUN_SCOPE}, same protocol={SAME_PROTOCOL}")
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
elif not TRAIN_MODEL:
    RUN_ID = f"{stamp}_seed{SEED}_load" + ("_controls" if RUN_ABLATIONS else "")
else:
    RUN_ID = f"{stamp}_seed{SEED}_{RUN_SCOPE}" + ("_mini" if IS_MINI_SAMPLE else "")
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_RUN_DIR = PERSIST_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_ID and PERSIST_RUN_DIR.is_dir() and not (RUN_DIR / "model_training").exists():
    shutil.copytree(PERSIST_RUN_DIR, RUN_DIR, dirs_exist_ok=True)
    print("Copied the Drive run folder locally for resumption.")

LOG_PATH = RUN_DIR / "logs" / "notebook_log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def log(*parts):
    message = " ".join(str(p) for p in parts)
    print(message)
    with open(LOG_PATH, "a") as handle:
        handle.write(time.strftime("%H:%M:%S ") + message + "\n")

M.set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "pyarrow": pa.__version__,
            "duckdb": installed("duckdb"), "cuda": torch.version.cuda, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
log("Run:", RUN_ID, "| local:", RUN_DIR, "| persistent:", PERSIST_RUN_DIR)
log("Versions:", VERSIONS)
if DEVICE.type != "cuda" and not SMOKE_TEST:
    log("WARNING: no GPU; full training will be slow. Runtime > Change runtime type > GPU.")

In [ ]:
# Optional: write verified copies of the vector files with nullable variable-length
# float32 lists (readable by any pyarrow). Not needed when Section 3's preflight
# succeeds, because the loader falls back to DuckDB for the known pyarrow 18-20
# fixed-size-list error. Values, dimensions, IDs, row order and nulls are compared
# exactly before a copy is accepted; originals are never modified.
VECTOR_COPIES = {}
if WRITE_COMPATIBLE_VECTORS:
    compat_root = Path(OUTPUT_ROOT) / "compatible_vectors"
    for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", 768),
                                    ("image", "image_embedding", "has_image_embedding", 1024)):
        for scope in ("train_val", "test"):
            key = f"{kind}_{scope}"
            destination = compat_root / scope / f"{kind}_embeddings.parquet"
            VECTOR_COPIES[key] = D.write_compatible_vectors(PATHS[key], destination,
                                                            column, flag, dim)
            PATH_OVERRIDES[key] = str(destination)
            log(key, {k: v for k, v in VECTOR_COPIES[key].items() if "sha" not in k})
    D.write_json(VECTOR_COPIES, compat_root / "compatible_vectors_manifest.json")
    PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT, PATH_OVERRIDES)
    log("Using verified compatible vector copies:", {k: str(PATHS[k]) for k in VECTOR_COPIES})
else:
    print("Using the original vector files (DuckDB fallback if pyarrow cannot decode them).")

## 3. Load and check shared data

**Candidate rows** come from the shared engineered table
`feature_engineering/{split}_features.parquet`. Only these columns are read:
- the keys and the label;
- `impression_time`;
- the request-context columns `hour`, `day_of_week`, `is_sso_user`,
  `is_subscriber`, `is_front_page` and the train-vocabulary device one-hots, all
  known before a recommendation is made and independent of the article pool;
- the shared `article_age_h`, only to check the catalogue-derived age.

Each impression is made contiguous; its in-view order is kept. No candidate is
added or removed. The 200 engineered history aggregates are deliberately **not**
inputs here. The model must learn from the ordered history itself, which also
keeps the attention-versus-summary comparison clean. `FEATURE_CASE` is recorded,
and the raw file is the only one read.

In [ ]:
import hashlib
FRAMES, OFFSETS, TABLE_INFO, SPLIT_START, REQUEST_SCOPE = {}, {}, {}, {}, {}
for split in D.SPLITS:
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    offsets = info.pop("offsets")
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())   # complete population
    if CHECK_AGAINST_BEHAVIOURS and PATHS.get(f"behaviors_{split}"):
        info["behaviour_cross_check"] = D.cross_check_behaviours(
            frame, PATHS[f"behaviors_{split}"], split)
        check = info["behaviour_cross_check"]
        assert check["feature_rows_not_in_behaviours"] == 0, check
        assert check["behaviour_candidates_not_in_features"] == 0, check
        assert check.get("label_mismatches", 0) == 0, check
    if split == "train":   # complete train population (counts, export IDs, full costs)
        first_all = frame.iloc[offsets[:-1]]
        TRAIN_POPULATION = {
            "user_id": first_all["user_id"].to_numpy(dtype=np.int64),
            "impression_id": first_all["impression_id"].to_numpy(dtype=np.int64),
            "time_us": first_all["impression_time"].astype("datetime64[us]").astype("int64").to_numpy()}
    if split == "train" and TRAIN_REQUESTS is not None:
        # Training requests: shared hash sample of the COMPLETE train population;
        # every recorded candidate of each selected request is kept.
        starts = offsets[:-1]
        first = frame.iloc[starts]
        chosen, train_bids = D.select_requests(
            first["user_id"].to_numpy(), first["impression_id"].to_numpy(),
            first["impression_time"].astype("datetime64[us]").astype("int64").to_numpy(),
            "train", SAMPLING_SEED, TRAIN_REQUESTS)
        frame, offsets = D.subset_requests_frame(frame, offsets, chosen)
        REQUEST_SCOPE["train"] = {"requests_selected": len(chosen),
                                  "requests_in_split": int(len(starts)),
                                  "candidate_rows": int(len(frame)),
                                  "ids_sha256": hashlib.sha256("\n".join(train_bids).encode()).hexdigest()}
        (RUN_DIR / "selection").mkdir(exist_ok=True)
        pd.DataFrame({"behaviour_id": train_bids}).to_csv(RUN_DIR / "selection" / "train_requests.csv",
                                                          index=False)
        if REFERENCE_TRAINING_IDS:
            reference = set(json.loads(Path(REFERENCE_TRAINING_IDS).read_text()))
            REFERENCE_TRAIN_MATCH = reference == set(train_bids)
            log("Training IDs identical to the 3.1 reference:", REFERENCE_TRAIN_MATCH)
            assert REFERENCE_TRAIN_MATCH, "training request IDs differ from the 3.1 reference"
    FRAMES[split], OFFSETS[split], TABLE_INFO[split] = frame, offsets, info

DEVICE_COLUMNS = TABLE_INFO["train"]["device_columns"]
for split in D.SPLITS:
    assert TABLE_INFO[split]["device_columns"] == DEVICE_COLUMNS, f"{split} device columns differ"
summary = pd.DataFrame({s: {k: v for k, v in TABLE_INFO[s].items()
                            if k not in ("device_columns", "behaviour_cross_check")}
                        for s in D.SPLITS}).T
display(summary)
for split in D.SPLITS:
    if "behaviour_cross_check" in TABLE_INFO[split]:
        log(split, "behaviour cross-check:", TABLE_INFO[split]["behaviour_cross_check"])
log("Run scope:", RUN_SCOPE, {"train_requests": TRAIN_REQUESTS, "selection_requests": SELECTION_REQUESTS,
                               "export_requests": EXPORT_REQUESTS, "sampling_seed": SAMPLING_SEED})
if "train" in REQUEST_SCOPE:
    log("Training requests (hash sample, complete slates):", REQUEST_SCOPE["train"])
assert TABLE_INFO["test"]["labelled"] is False

**Article catalogue and frozen vectors (preflight).**
- **Catalogue.** The train/validation catalogue takes precedence; the test
  catalogue adds test-only articles (first-wins, as in Step 3).
- **Vectors.** The vector columns are actually decoded. The checks:
  - dimensions (768 text / 1024 image);
  - finite values;
  - flag ↔ null agreement;
  - duplicate IDs;
  - identical vectors for IDs present in both lookups.
- **Missing vs unreadable.** A legitimately missing vector stays missing (with a
  flag). An unreadable file stops the notebook with a diagnosis; it is not
  replaced with zeros.

In [ ]:
VOCAB = json.loads(PATHS["feature_vocabulary"].read_text())
CATALOGUE, CATALOGUE_META = D.load_catalogue([PATHS["articles_train_val"], PATHS["articles_test"]])
CATALOGUE_INFO = CATALOGUE_META["info"]   # ordered union: train_val then test, first record wins
SNAPSHOT_IDS = {"train_val": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_train_val"])],
                "test": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_test"])]}
log("Catalogue:", CATALOGUE_INFO)

def expected_dim(kind, default):
    for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
        if PATHS.get(key):
            meta = json.loads(PATHS[key].read_text())
            value = meta.get(f"{kind}_dimension")
            if value and value != default:
                raise ValueError(f"{key}: {kind} dimension {value} != {default}")
    return default
TEXT_DIM, IMAGE_DIM = expected_dim("text", 768), expected_dim("image", 1024)

VECTOR_INFO, VECTOR_SOURCES = {}, {"text": [], "image": []}
for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", TEXT_DIM),
                                ("image", "image_embedding", "has_image_embedding", IMAGE_DIM)):
    for scope in ("train_val", "test"):
        ids, matrix, available, info = D.decode_vectors(PATHS[f"{kind}_{scope}"], column, flag, dim)
        VECTOR_SOURCES[kind].append((ids, matrix, available))
        VECTOR_INFO[f"{kind}_{scope}"] = info
        log(f"{kind:5s} {scope:9s}", {k: v for k, v in info.items() if k != "path"})
TEXT_LOOKUP, text_union = D.union_vectors(VECTOR_SOURCES["text"])
IMAGE_LOOKUP, image_union = D.union_vectors(VECTOR_SOURCES["image"])
for name, union in (("text", text_union), ("image", image_union)):
    assert union["overlap_vector_mismatches"] == 0, f"{name}: overlapping IDs disagree"
    VECTOR_INFO[f"{name}_union"] = {k: v for k, v in union.items() if k != "known_ids"}
    log(name, "union:", VECTOR_INFO[f"{name}_union"])
del VECTOR_SOURCES

**Chronological histories.** For each split, its own fixed `history.parquet`:
- **Alignment.** The four lists (article ID, timestamp, read time, scroll %) must
  have equal lengths.
- **Invalid events.** Events with a null ID or timestamp are dropped and counted.
  Invalid read/scroll values become missing, with flags.
- **Cutoff.** Every event must be strictly before the split's earliest impression;
  otherwise the notebook stops.
- **Ordering.** Events are sorted chronologically; the original list position
  breaks ties. Repeat reads of the same article are kept as separate events.
- **Truncation.** The most recent `HISTORY_MAX_LEN` events are kept, oldest
  first, then right-padded with masks.
- **Users without a history row.** These map to an empty row and the model's
  empty-history fallback.

In [ ]:
HISTORIES = {}
for split in D.SPLITS:
    # Cutoff: the split's earliest request in the complete population.
    HISTORIES[split] = D.build_history(PATHS[f"history_{split}"], SPLIT_START[split],
                                       HISTORY_MAX_LEN, FRAMES[split]["user_id"].unique())
    log(split, "history:", HISTORIES[split].stats)

## 4. Prepare model inputs

**Fitted on train only** (after any development sampling):
- standardisation constants for log candidate age, log history length, log gap
  to request, log read time, scroll fraction, inter-event gap, body length and
  sentiment;
- the optional article-ID vocabulary.

**Reused unchanged:** the shared Step 3 train vocabulary (category, article type,
topics).

In load-only mode every fitted value is read from the saved bundle; nothing is
refitted.

In [ ]:
def referenced_ids(split):
    hist = HISTORIES[split]
    mask = np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]
    return np.concatenate([FRAMES[split]["candidate_article_id"].to_numpy(),
                           hist.article_ids[mask]])

# The article table covers every catalogue article (both snapshots) plus any
# other referenced ID, so catalogue requests can be scored.
all_ids = np.concatenate([referenced_ids(s) for s in D.SPLITS] + [CATALOGUE.index.to_numpy()])
if TRAIN_MODEL:
    train_ids = referenced_ids("train")
    article_stats = D.fit_article_stats(CATALOGUE, train_ids)
    ID_VOCAB = (D.fit_id_vocab(train_ids, ID_EMBEDDING_MIN_COUNT)
                if MODEL_HPARAMS["use_id_embedding"] else {})
    ARTICLES, ARTICLE_INFO = D.build_article_table(
        CATALOGUE, all_ids, VOCAB, article_stats, TEXT_LOOKUP, IMAGE_LOOKUP,
        TEXT_DIM, IMAGE_DIM, 8, ID_VOCAB)
    PREPROCESSING = {
        "numeric_stats": D.fit_numeric_stats(FRAMES["train"], HISTORIES["train"],
                                             OFFSETS["train"], ARTICLES),
        "article_stats": article_stats,
        "history_max_len": HISTORY_MAX_LEN, "device_columns": DEVICE_COLUMNS,
        "feature_vocabulary": VOCAB, "text_dim": TEXT_DIM, "image_dim": IMAGE_DIM,
        "max_topics": 8, "shuffle_seed": SHUFFLE_SEED,
        "candidate_features": D.CANDIDATE_FEATURE_NAMES,
    }
    seen_train_ids = set(np.unique(train_ids).tolist())
else:
    PREPROCESSING = json.loads((LOAD_DIR / "preprocessing.json").read_text())
    mappings = np.load(LOAD_DIR / "mappings.npz")
    ID_VOCAB = dict(zip(mappings["id_vocab_ids"].tolist(), mappings["id_vocab_rows"].tolist()))
    seen_train_ids = set(mappings["train_referenced_ids"].tolist())
    assert HISTORIES["train"].article_ids.shape[1] == PREPROCESSING["history_max_len"]
    assert PREPROCESSING["feature_vocabulary"] == VOCAB, "shared vocabulary changed since training"
    assert PREPROCESSING["device_columns"] == DEVICE_COLUMNS, "device columns changed"
    assert PREPROCESSING.get("candidate_features") == D.CANDIDATE_FEATURE_NAMES, \
        "saved model uses a different candidate feature set; retrain"
    ARTICLES, ARTICLE_INFO = D.build_article_table(
        CATALOGUE, all_ids, VOCAB, PREPROCESSING["article_stats"], TEXT_LOOKUP, IMAGE_LOOKUP,
        TEXT_DIM, IMAGE_DIM, PREPROCESSING["max_topics"], ID_VOCAB)
    log("Loaded preprocessing from", LOAD_DIR)
log("Article table:", ARTICLE_INFO)
assert ARTICLES.numeric_names[MODEL_HPARAMS.get("numeric_has_text_index", 7)] == "has_text_vector"

def make_arrays(variant, splits=D.SPLITS):
    # Ordered arrays; for the order control, the same arrays with each user's
    # events permuted after all per-event features were computed chronologically.
    out = {}
    for split in splits:
        arrays = D.build_split_arrays(split, FRAMES[split], OFFSETS[split], HISTORIES[split],
                                      ARTICLES, PREPROCESSING["numeric_stats"], DEVICE_COLUMNS)
        if variant == "attention_shuffled":
            arrays = D.permute_history(arrays, PREPROCESSING["shuffle_seed"] + D.SPLITS.index(split))
        out[split] = arrays
    return out

ARRAYS = {"ordered": make_arrays("attention")}
MAIN_KEY = "shuffled" if MODEL_VARIANT == "attention_shuffled" else "ordered"
if MAIN_KEY not in ARRAYS:
    ARRAYS[MAIN_KEY] = make_arrays(MODEL_VARIANT)
coverage = {s: D.coverage_report(ARRAYS["ordered"][s], ARTICLES) for s in D.SPLITS}
display(pd.DataFrame({(s, part): v for s in coverage for part, v in coverage[s].items()}).T)
for split in D.SPLITS:
    a = ARRAYS["ordered"][split]
    assert np.isfinite(a.imp_context).all() and np.isfinite(a.cand_features).all()
    assert np.isfinite(a.hist_features).all()
    assert len(a.cand_article_id) == len(FRAMES[split])
log("Context features:", ARRAYS["ordered"]["train"].context_names)
log("Candidate features:", ARRAYS["ordered"]["train"].candidate_names)
log("Event features:", ARRAYS["ordered"]["train"].event_names)
assert ARRAYS["ordered"]["train"].event_names[M.GAP_PREV_FEATURE] == "log_gap_prev_h_z"
assert ARRAYS["ordered"]["train"].event_names[M.GAP_PREV_MISSING] == "gap_prev_missing"

# Catalogue-derived candidate age vs the shared table's article_age_h.
AGE_CHECK = {}
for split in D.SPLITS:
    a = ARRAYS["ordered"][split]
    ours = D.candidate_age_hours(ARTICLES.published_s[a.cand_article_row],
                                 ARTICLES.published_missing[a.cand_article_row],
                                 np.repeat(a.imp_time_s, np.diff(a.offsets)))
    shared = FRAMES[split]["article_age_h"].to_numpy(dtype="float64", na_value=np.nan)
    both = np.isfinite(ours) & np.isfinite(shared)
    AGE_CHECK[split] = {"missing_disagreements": int((np.isfinite(ours) != np.isfinite(shared)).sum()),
                        "max_abs_diff_hours": float(np.abs(ours[both] - shared[both]).max())
                        if both.any() else None}
log("Candidate age vs shared article_age_h:", AGE_CHECK)
for split, check in AGE_CHECK.items():
    assert check["missing_disagreements"] == 0 and (check["max_abs_diff_hours"] or 0) < 0.01, split

### Catalogue pool, request IDs and protocol fingerprint

**Pool policy `published_at_or_before_request_v1`** (shared with 3.1):
- **One catalogue for every split:** the ordered union of
  `train_val/articles.parquet` then `test/articles.parquet`. The **first** record
  per article ID is kept, including its missing fields.
- **Eligibility:** an article is eligible when its publication time is known
  and at or before the request time (exact microsecond comparison).
- **No other filters:** there is no age or access cutoff and no requirement for
  an embedding. Articles without vectors stay eligible, with missing flags.
- **Full pools:** every selected request is scored against **all** of its
  eligible articles. Only requests are ever sampled, never articles.

**Request IDs (shared rule):**
- `behaviour_id = f"{user_id}:{impression_id}"`.
- Each request's priority is `blake2b(f"{seed}:{split}:{behaviour_id}", digest_size=8)`.
- The N smallest by (priority, behaviour_id) are selected from the **complete**
  request population.
- Selected requests are processed in (timestamp, user_id, impression_id) order.

**Limits:** training, selection-validation and export limits are independent.
With `aligned_dev`, the 25 exported validation requests are the 25
lowest-priority ones, so they are a subset of the 100 selection requests. The two
metric sets are still reported separately.

**Protocol fingerprint:** the pool policy, sources, sample IDs and encoder identity
are stored with every checkpoint, so a run under another protocol is never
resumed as this one.

In [ ]:
UNION_IDS = CATALOGUE.index.to_numpy()            # ordered union, first record per ID
def portable(path):
    # Paths relative to DATA_ROOT, so fingerprints survive a different Drive mount.
    try:
        return str(Path(path).resolve().relative_to(DATA_ROOT.resolve()))
    except ValueError:
        return str(path)

POLICY = D.EligibilityPolicy([portable(PATHS["articles_train_val"]), portable(PATHS["articles_test"])])
union_index = D.build_catalogue_index("all", "union", UNION_IDS, ARTICLES)
CAT_INDEX = {s: union_index for s in D.SPLITS}    # the SAME pool for every split
ROW_BYTES_GUESS = 64.0   # replaced by a measured value in Section 7
ELIGIBILITY = {s: D.eligibility_summary(ARRAYS["ordered"][s], CAT_INDEX[s], None, ROW_BYTES_GUESS)
               for s in D.SPLITS}
display(pd.DataFrame(ELIGIBILITY).T)
CLICK_ELIGIBILITY = {s: D.clicked_eligibility(ARRAYS["ordered"][s], CAT_INDEX[s], ARTICLES)
                     for s in D.SPLITS}
for split, info in CLICK_ELIGIBILITY.items():
    if info.get("labelled"):
        log(split, "known clicks vs eligibility:", info["summary"])
log("Union catalogue:", {"articles": int(len(UNION_IDS)),
                         "excluded_unknown_publication": union_index.excluded_missing_published,
                         "with_text_vector": int(union_index.cum_text[-1]),
                         "with_image_vector": int(union_index.cum_image[-1])})

def request_ids(split, limit):
    a = ARRAYS["ordered"][split]
    if split == "train" and TRAIN_REQUESTS is not None:
        # Select from the COMPLETE train population, then map into the training subset.
        if limit is None or limit > TRAIN_REQUESTS:
            raise ValueError("train export requests must be a subset of the training requests")
        _, bids = D.select_requests(TRAIN_POPULATION["user_id"], TRAIN_POPULATION["impression_id"],
                                    TRAIN_POPULATION["time_us"], "train", SAMPLING_SEED, limit)
        position = {b: i for i, b in enumerate(D.behaviour_ids(a.user_id, a.impression_id))}
        return np.asarray([position[b] for b in bids], dtype=np.int64), bids
    return D.select_requests(a.user_id, a.impression_id, a.imp_time_us, split, SAMPLING_SEED, limit)

def population_size(split):
    return int(len(TRAIN_POPULATION["user_id"])) if split == "train" else int(ARRAYS["ordered"][split].n_impressions)

def ids_digest(bids):
    return hashlib.sha256("\n".join(bids).encode()).hexdigest()

# Selection-validation requests (from the COMPLETE validation population).
SELECTION_SAMPLE, selection_bids = request_ids("validation", SELECTION_REQUESTS)
REFERENCE_SELECTION_MATCH = None
if REFERENCE_INPUT_AUDIT:
    reference = json.loads(Path(REFERENCE_INPUT_AUDIT).read_text()).get("selection_requests")
    REFERENCE_SELECTION_MATCH = reference == selection_bids
    log("Selection IDs identical to the 3.1 reference:", REFERENCE_SELECTION_MATCH)
    assert REFERENCE_SELECTION_MATCH, "selection requests differ from 3.1's input_audit.json"
# Export requests per split (independent limit).
REQUESTS, LIMITED, EXPORT_BIDS = {}, {}, {}
for split in D.SPLITS:
    REQUESTS[split], EXPORT_BIDS[split] = request_ids(split, EXPORT_REQUESTS)
    LIMITED[split] = len(REQUESTS[split]) < population_size(split)
(RUN_DIR / "selection").mkdir(exist_ok=True)
pd.DataFrame({"behaviour_id": selection_bids}).to_csv(RUN_DIR / "selection" / "validation_requests.csv",
                                                      index=False)
for split in D.SPLITS:
    pd.DataFrame({"behaviour_id": EXPORT_BIDS[split]}).to_csv(
        RUN_DIR / "selection" / f"{split}_export_requests.csv", index=False)
REQUEST_SCOPE.update({
    "run_scope": RUN_SCOPE, "sampling_seed": SAMPLING_SEED,
    "rule": "smallest (blake2b-64('seed:split:behaviour_id'), behaviour_id) from the complete "
            "request population; processed by (timestamp, user_id, impression_id); full pools",
    "train_requests": TRAIN_REQUESTS, "selection_requests": SELECTION_REQUESTS,
    "export_requests": EXPORT_REQUESTS,
    "selection": {"count": int(len(SELECTION_SAMPLE)), "ids_sha256": ids_digest(selection_bids),
                  "file": "selection/validation_requests.csv"},
    "export": {s: {"count": int(len(REQUESTS[s])), "of_total": population_size(s),
                   "sampled": LIMITED[s], "ids_sha256": ids_digest(EXPORT_BIDS[s]),
                   "file": f"selection/{s}_export_requests.csv"} for s in D.SPLITS},
    "exported_validation_subset_of_selection": set(EXPORT_BIDS["validation"]) <= set(selection_bids),
})
display(pd.DataFrame({s: D.eligibility_summary(ARRAYS["ordered"][s], CAT_INDEX[s], None, ROW_BYTES_GUESS,
                                               requests=REQUESTS[s]) for s in D.SPLITS}).T)

TEXT_ENCODER = {"model": None, "dimension": TEXT_DIM, "source": "embedding metadata.json"}
for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
    if PATHS.get(key):
        meta = json.loads(PATHS[key].read_text())
        TEXT_ENCODER["model"] = TEXT_ENCODER["model"] or meta.get("text_model")
        TEXT_ENCODER.setdefault("image_source", meta.get("image_source"))
if RUN_SCOPE == "full" and not TEXT_ENCODER["model"]:
    log("WARNING: text encoder identity unknown; confirm it with the preprocessing owner.")

PROTOCOL = {
    "version": "lstm_catalogue_protocol_v2", "pool_policy": POLICY.to_json(),
    "catalogue_sources": {k: {"path": portable(PATHS[k]), "bytes": PATHS[k].stat().st_size,
                              "mtime_ns": PATHS[k].stat().st_mtime_ns}
                          for k in ("articles_train_val", "articles_test")},
    "vector_sources": {k: {"path": portable(PATHS[k]), "bytes": PATHS[k].stat().st_size,
                           "mtime_ns": PATHS[k].stat().st_mtime_ns}
                       for k in ("text_train_val", "text_test", "image_train_val", "image_test")},
    "vector_union_rule": "train_val then test; first source record per ID wins, even if missing",
    "union_catalogue_articles": int(len(UNION_IDS)),
    "union_catalogue_ids_sha256": hashlib.sha256(np.sort(UNION_IDS).astype("<i8").tobytes()).hexdigest(),
    "duplicate_precedence": "first source record (train_val before test)",
    "request_scope": dict(REQUEST_SCOPE),
    "text_encoder": TEXT_ENCODER, "image_vectors": "EB-NeRD supplied, 1024-d",
}

SELECTION = {
    "version": 2, "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
    "direction": "maximize", "final_tiebreak": "earlier checkpoint / trial",
    "interval": SELECTION_INTERVAL, "patience_counts": "selection evaluations",
    "min_improvement": "strictly greater (objective, tiebreak) tuple",
    "partition": "validation (development)", "pool_policy": POLICY.to_json(),
    "shortlist_size": SHORTLIST_SIZE,
    "requests": {"count": int(len(SELECTION_SAMPLE)),
                 "of_total": int(ARRAYS["ordered"]["validation"].n_impressions),
                 "seed": SAMPLING_SEED, "rule": REQUEST_SCOPE["rule"],
                 "file": "selection/validation_requests.csv",
                 "behaviour_ids_sha256": ids_digest(selection_bids)},
    "protocol_sha256": hashlib.sha256(json.dumps(PROTOCOL, sort_keys=True, default=str).encode()).hexdigest(),
    "model_inputs": {"history_max_len": HISTORY_MAX_LEN, "model_variant": MODEL_VARIANT},
    "logged_metrics": [SELECTION_OBJECTIVE, SELECTION_TIEBREAK, "eligible_only_recall@100",
                       "requests_with_known_clicks_evaluated", "ties_at_cutoff",
                       "diagnostic_recorded_slate_auc"],
}
log("Selection declaration:", {k: v for k, v in SELECTION.items() if k != "pool_policy"})
assert SELECTION_OBJECTIVE.endswith(f"@{SHORTLIST_SIZE}") and SELECTION_TIEBREAK.endswith(f"@{SHORTLIST_SIZE}"), \
    "selection metric names must match SHORTLIST_SIZE"
REQUEST_SCOPE["selection"]["matches_reference"] = REFERENCE_SELECTION_MATCH   # after hashing
if not TRAIN_MODEL and SAME_PROTOCOL:
    # The whole protocol (pool, sources, union IDs, train/selection/export IDs, encoder)
    # must match the weights' protocol before controls/evaluation are compared to it.
    assert SOURCE_SELECTION.get("protocol_sha256") == SELECTION["protocol_sha256"], \
        "protocol differs from the loaded run's (pool, sources, request IDs or encoder)"
if TRAIN_MODEL:
    WEIGHTS_SELECTED_UNDER = {"protocol_version": PROTOCOL["version"],
                              "protocol_sha256": SELECTION["protocol_sha256"],
                              "objective": SELECTION_OBJECTIVE, "trained_in_run": RUN_ID}

In [ ]:
# Device tensors (article vectors stay in one lookup table; candidates hold indices).
ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
TENSORS = {key: {s: M.SplitTensors(ARRAYS[key][s], DEVICE) for s in D.SPLITS}
           for key in ARRAYS}
VAL = ARRAYS["ordered"]["validation"]

def evaluate_validation(scores, arrays=VAL, per_impression=False):
    return D.impression_metrics(arrays.offsets, arrays.labels, scores, k=5,
                                return_per_impression=per_impression)

def model_config(variant, overrides=None):
    sample = ARRAYS["ordered"]["train"]
    stats = PREPROCESSING["numeric_stats"]["log_gap_to_impression_h"]
    params = dict(MODEL_HPARAMS, **(overrides or {}))
    params["head_hidden"] = tuple(params["head_hidden"])
    return M.ModelConfig(
        variant=variant, n_categories=len(VOCAB["category"]) + 3,
        n_types=len(VOCAB["article_type"]) + 3, n_topics=len(VOCAB["topics"]) + 2,
        n_article_numeric=ARTICLES.numeric.shape[1], text_dim_in=TEXT_DIM, image_dim_in=IMAGE_DIM,
        n_id_buckets=max(len(ID_VOCAB) + 1, 1), n_event_features=sample.hist_features.shape[-1],
        n_candidate_features=sample.cand_features.shape[1],
        n_context_features=sample.imp_context.shape[1],
        gap_mean=stats["mean"], gap_std=stats["std"], **params)

VAL_CLICK_LOOKUP = D.clicks_by_request(CLICK_ELIGIBILITY["validation"])

SELECTION_SLATES = {}

def make_selector(key, requests=None):
    # Selection callback: exact top-K over the FULL eligible validation catalogue for
    # each sampled request (fresh article/user encodings for the current weights),
    # plus the recorded-slate AUC as a diagnostic. Unknown outcomes stay unknown.
    requests = SELECTION_SAMPLE if requests is None else np.asarray(requests)
    def select_fn(m, return_per_request=False):
        scorer = M.CatalogueScorer(m, TENSORS[key]["validation"], CAT_INDEX["validation"],
                                   PREPROCESSING["numeric_stats"]["log_candidate_age_h"],
                                   MAX_ARTICLE_AGE_HOURS, CATALOGUE_CHUNK, requests=requests)
        shortlists, ties = M.catalogue_shortlists(scorer, requests, SHORTLIST_SIZE)
        del scorer
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        hits = {i: h for i, top in shortlists.items()
                if (h := D.click_hits(VAL_CLICK_LOOKUP, i, top, (SHORTLIST_SIZE,))) is not None}
        cat = D.recall_from_hits(hits, CLICK_ELIGIBILITY["validation"], (SHORTLIST_SIZE,),
                                 requests_scored=len(requests))
        if key not in SELECTION_SLATES:
            sub = D.subset_split_arrays(ARRAYS[key]["validation"], requests)
            SELECTION_SLATES[key] = (sub, M.SplitTensors(sub, DEVICE))
        sub, sub_tensors = SELECTION_SLATES[key]
        slate = evaluate_validation(M.predict_logits(m, sub_tensors), arrays=sub,
                                    per_impression=return_per_request)
        slate["scope"] = "recorded slates of the selection requests"
        out = {k: v for k, v in cat.items() if not isinstance(v, dict)}
        out.update({"ties_at_cutoff": ties, "requests_scored": int(len(requests)),
                    "diagnostic_recorded_slate_auc": slate["mean_impression_auc"]})
        if return_per_request:
            out["_per_request"] = D.per_request_metrics(hits, requests, SHORTLIST_SIZE)
            out["_slate"] = slate
        return out
    return select_fn

if DEVICE.type == "cuda":
    log(f"GPU memory allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

## 5. Define and train / load model

**Architecture** (see `lstm_model.py`):
1. **Article encoder**, shared by history events and candidates. It concatenates:
   - category, type and topic embeddings (shared train vocabulary);
   - numeric metadata and missing flags;
   - a projection of the frozen text vector;
   - a separate projection of the frozen image vector.

   An MLP and LayerNorm then give a 128-d article vector. The supplied vectors are
   frozen lookups; only the projections train. During training only, 20% of
   candidate articles have their text vector hidden (flag set to "no text").
2. **History encoder.** Each event is the article vector plus log read time,
   scroll and missing flags, log gap since the previous event, and log article age
   when read. Events pass through a linear+ReLU layer, then a 1-layer LSTM (packed
   sequences, oldest → newest). Padding is masked.
3. **Target-aware attention.** The candidate vector becomes the query and the LSTM
   states the keys. The score is the scaled dot product plus a learned
   recency bias × standardised log(hours from event to impression). A masked
   softmax gives weights, and the candidate-specific context is the weighted sum of
   states. With an empty history the model uses a learned fallback vector instead.
4. **Head.** An MLP (256 → 128 → 1, dropout) over:
   - the candidate vector and the context;
   - the elementwise product of their projections and its scaled dot product;
   - candidate freshness (log article age at the request);
   - request context (time, device, subscriber, front page, history length).

   Output: a click logit. Training loss is BCE over every candidate.

The course recipe regressed movie ratings with SmoothL1; here the target is a
binary click, so the loss is `BCEWithLogitsLoss`, and `sigmoid(logit)` is exported
as an uncalibrated click probability.

In [ ]:
MAIN_CONFIG = model_config(MODEL_VARIANT)
_preview = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
log("Main model:", MAIN_CONFIG.variant, M.count_parameters(_preview))
print(_preview)
del _preview

### Optional: bounded Optuna search (off by default)

- **Objective:** maximise validation known-click Recall@100 (the same fixed request
  sample and full pools as checkpoint selection); ties go to observed-click
  nDCG@100, then the earlier trial. Development data; test is never touched.
  Test is never touched.
- **Sampler and pruner:** TPE (seeded) with a MedianPruner over epochs.
- **Budget:** `OPTUNA_N_TRIALS` trials, each at most `OPTUNA_EPOCHS` epochs on
  `OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH` sampled training impressions.
- **Storage:** a local SQLite file. A consistent snapshot (SQLite backup API) and
  `trials.csv` are copied to Drive after every trial.
- **Resuming:** rerunning copies the Drive snapshot back and continues the study.
  Trials left `RUNNING` by a crash stay marked as such.
- **Not searched:** history length, because it would require rebuilding the
  sequences.

In [ ]:
TUNING_SUMMARY = None
if RUN_OPTUNA and TRAIN_MODEL:
    try:
        import optuna
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
        import optuna
    import sqlite3
    tuning_local = Path("/content/optuna" if IN_COLAB else RUN_DIR / "optuna_local")
    tuning_local.mkdir(parents=True, exist_ok=True)
    tuning_drive = PERSIST_ROOT / "tuning" / OPTUNA_STUDY_NAME
    db_local = tuning_local / f"{OPTUNA_STUDY_NAME}.db"
    db_drive = tuning_drive / f"{OPTUNA_STUDY_NAME}.db"
    if db_drive.is_file() and not db_local.is_file():
        shutil.copy2(db_drive, db_local)
        log("Resuming study from Drive snapshot", db_drive)

    def snapshot(study, trial=None):
        tuning_drive.mkdir(parents=True, exist_ok=True)
        tmp = tuning_local / "snapshot.db"
        with sqlite3.connect(db_local) as src, sqlite3.connect(tmp) as dst:
            src.backup(dst)
        shutil.copy2(tmp, db_drive)
        study.trials_dataframe().to_csv(tuning_drive / "trials.csv", index=False)

    def objective(trial):
        overrides = dict(
            lstm_hidden=trial.suggest_categorical("lstm_hidden", [64, 128, 256]),
            attention_dim=trial.suggest_categorical("attention_dim", [32, 64, 128]),
            article_dim=trial.suggest_categorical("article_dim", [64, 128]),
            dropout=trial.suggest_float("dropout", 0.1, 0.4),
        )
        train_over = dict(
            learning_rate=trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
            weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            history_dropout=trial.suggest_float("history_dropout", 0.0, 0.3),
        )
        M.set_seed(SEED + trial.number)
        model = M.build_model(model_config(MODEL_VARIANT, overrides), ARTICLE_TENSORS, DEVICE)
        cfg = M.TrainConfig(**dict(TRAIN_HPARAMS, **train_over, max_epochs=OPTUNA_EPOCHS,
                                   max_train_impressions_per_epoch=OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH,
                                   seed=SEED + trial.number))
        try:
            _, best = M.train_model(model, TENSORS[MAIN_KEY]["train"], make_selector(MAIN_KEY),
                                    SELECTION, cfg, tuning_local / f"trial_{trial.number}",
                                    resume=False, trial=trial, log=log)
        finally:   # also removes pruned/failed trial folders
            shutil.rmtree(tuning_local / f"trial_{trial.number}", ignore_errors=True)
        trial.set_user_attr("best_epoch", best["epoch"])
        trial.set_user_attr(SELECTION_TIEBREAK, best["key"][1])
        return best["key"][0]

    study = optuna.create_study(
        study_name=OPTUNA_STUDY_NAME, storage=f"sqlite:///{db_local}", direction="maximize",
        load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1))
    declaration = json.loads(json.dumps(SELECTION, default=str))
    if "selection" in study.user_attrs:
        if study.user_attrs["selection"] != declaration:
            raise RuntimeError(f"Study {OPTUNA_STUDY_NAME} was run under a different selection "
                               "declaration; use a new OPTUNA_STUDY_NAME.")
    elif study.trials:
        raise RuntimeError(f"Study {OPTUNA_STUDY_NAME} has trials but no selection declaration "
                           "(e.g. an older AUC study); use a new OPTUNA_STUDY_NAME.")
    else:
        study.set_user_attr("selection", declaration)
    remaining = OPTUNA_N_TRIALS - len([t for t in study.trials if t.state.is_finished()])
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, timeout=OPTUNA_TIMEOUT_MIN * 60,
                       callbacks=[snapshot], gc_after_trial=True)
    snapshot(study)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    # Contract tie rule: Recall@100, then nDCG@100, then the earlier trial.
    best_trial = max(completed, key=lambda t: (t.value, t.user_attrs.get(SELECTION_TIEBREAK, -1.0),
                                               -t.number))
    TUNING_SUMMARY = {
        "study_name": OPTUNA_STUDY_NAME, "objective": f"max validation {SELECTION_OBJECTIVE}, tie {SELECTION_TIEBREAK}, then earlier trial "
        "(development data)", "sampler": "TPESampler(seed=SEED)",
        "pruner": "MedianPruner(n_startup_trials=3, n_warmup_steps=1)",
        "budget": {"n_trials": OPTUNA_N_TRIALS, "timeout_min": OPTUNA_TIMEOUT_MIN,
                   "epochs": OPTUNA_EPOCHS,
                   "train_impressions_per_epoch": OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH},
        "selection_sample": SELECTION["requests"],
        "best_value": best_trial.value, "best_tiebreak": best_trial.user_attrs.get(SELECTION_TIEBREAK),
        "best_params": best_trial.params, "best_trial": best_trial.number,
        "trial_states": {str(st): sum(t.state == st for t in study.trials)
                         for st in {t.state for t in study.trials}},
        "snapshot": str(db_drive),
        "trials_csv": str(tuning_drive / "trials.csv"), "optuna_version": optuna.__version__,
    }
    log("Optuna:", TUNING_SUMMARY)
    display(study.trials_dataframe())
    if APPLY_BEST_TRIAL:
        p = best_trial.params
        for key in ("lstm_hidden", "attention_dim", "article_dim", "dropout"):
            MODEL_HPARAMS[key] = p[key]
        for key in ("learning_rate", "weight_decay", "history_dropout"):
            TRAIN_HPARAMS[key] = p[key]
        MAIN_CONFIG = model_config(MODEL_VARIANT)
        log("Applied best trial to MODEL_HPARAMS/TRAIN_HPARAMS")
else:
    print("Optuna search skipped (RUN_OPTUNA=False or load-only mode).")

### Interrupted-run / resume check (`RUN_RESUME_CHECK`)

This check trains two throwaway models on a few batches, in a separate
`resume_check/` folder:
- **Run A** trains 2 epochs without interruption.
- **Run B** is interrupted after epoch 1 by a simulated crash. Its recovery
  checkpoint is copied to Drive and back, as a new runtime would do. It is then
  resumed into a *differently initialised* model object.

What it checks:
- the restored RNG and generator states can be applied after the checkpoint was
  loaded with `map_location` = GPU;
- the resumed run continues at epoch 2;
- the optimizer state ends on the training device;
- how closely run B's final weights and validation logits match run A's. This is
  reported but not required, because cuDNN/AMP are not bitwise deterministic.

Neither model is used for exports.

In [ ]:
class SimulatedInterrupt(Exception):
    pass

RESUME_CHECK = None
if RUN_RESUME_CHECK and TRAIN_MODEL and not RESUME_RUN_ID:
    rc_root = RUN_DIR / "resume_check"
    shutil.rmtree(rc_root, ignore_errors=True)
    rc_cfg = M.TrainConfig(**dict(TRAIN_HPARAMS, max_epochs=2, early_stopping_patience=5,
                                  max_train_impressions_per_epoch=4 * TRAIN_HPARAMS["batch_impressions"],
                                  seed=SEED))
    tr, va = TENSORS[MAIN_KEY]["train"], TENSORS[MAIN_KEY]["validation"]
    rc_requests = SELECTION_SAMPLE[:20]
    rc_select = make_selector(MAIN_KEY, rc_requests)
    rc_selection = {**SELECTION, "requests": {"count": int(len(rc_requests)),
                                              "note": "resume check only (first 20 sampled)"}}
    quiet = lambda *a: None
    M.set_seed(SEED)
    model_a = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    hist_a, _ = M.train_model(model_a, tr, rc_select, rc_selection, rc_cfg, rc_root / "A",
                              resume=False, log=quiet)

    def interrupt_after_first(epoch, row):
        if epoch == 1:
            raise SimulatedInterrupt()
    M.set_seed(SEED)
    model_b = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    try:
        M.train_model(model_b, tr, rc_select, rc_selection, rc_cfg, rc_root / "B",
                      resume=False, on_epoch_end=interrupt_after_first, log=quiet)
        raise AssertionError("the simulated interruption did not happen")
    except SimulatedInterrupt:
        pass
    del model_b
    drive_trip = None
    if PERSIST_TO_DRIVE:   # emulate a new runtime: recovery files come back from Drive
        shutil.copytree(rc_root / "B", PERSIST_RUN_DIR / "resume_check" / "B", dirs_exist_ok=True)
        shutil.rmtree(rc_root / "B")
        shutil.copytree(PERSIST_RUN_DIR / "resume_check" / "B", rc_root / "B")
        drive_trip = str(PERSIST_RUN_DIR / "resume_check" / "B")

    # The risky path: a checkpoint mapped onto the GPU must still restore RNG state.
    mapped = torch.load(rc_root / "B" / "recovery" / "last_checkpoint.pt",
                        map_location=DEVICE, weights_only=False)
    M.restore_rng_state(mapped["rng"])
    torch.Generator().set_state(M._cpu_byte_tensor(mapped["generator"]))
    rng_devices = {"torch_state_loaded_on": str(mapped["rng"]["torch"].device),
                   "generator_state_loaded_on": str(mapped["generator"].device)}
    del mapped

    M.set_seed(SEED + 12345)   # different initial weights: resumption must overwrite them
    model_c = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    messages = []
    hist_c, _ = M.train_model(model_c, tr, rc_select, rc_selection, rc_cfg, rc_root / "B",
                              resume=True, log=messages.append)
    recovery_state = torch.load(rc_root / "B" / "recovery" / "last_checkpoint.pt",
                                map_location="cpu", weights_only=False)
    final_a = torch.load(rc_root / "A" / "recovery" / "last_checkpoint.pt",
                         map_location="cpu", weights_only=False)["model"]
    param_diff = max(float((final_a[k].float() - recovery_state["model"][k].float()).abs().max())
                     for k in final_a)
    model_a.load_state_dict({k: v.to(DEVICE) for k, v in final_a.items()})   # last epoch, as for B
    logits_a = M.predict_logits(model_a, va)
    model_c.load_state_dict({k: v.to(DEVICE) for k, v in recovery_state["model"].items()})
    logits_c = M.predict_logits(model_c, va)
    opt_devices = {str(v.device) for st in recovery_state["optimizer"]["state"].values()
                   for v in st.values() if torch.is_tensor(v) and v.dim() > 0}
    RESUME_CHECK = {
        "device": str(DEVICE), "epochs_A": [r["epoch"] for r in hist_a],
        "epochs_resumed_B": [r["epoch"] for r in hist_c],
        "resumed_message": next((m for m in messages if m.startswith("Resumed")), None),
        "drive_round_trip": drive_trip, **rng_devices,
        "optimizer_state_devices_in_checkpoint": sorted(opt_devices),
        "final_param_max_abs_diff_vs_uninterrupted": param_diff,
        "validation_logit_max_abs_diff_vs_uninterrupted": float(np.abs(logits_a - logits_c).max()),
        "close_to_uninterrupted_1e-5": bool(param_diff <= 1e-5),
    }
    # Model + optimizer state must really be restored: the resumed weights must match
    # the uninterrupted run up to GPU nondeterminism (a failed restore differs by
    # far more, because run C started from different random weights).
    RESUME_CHECK["param_tolerance"] = 1e-3
    RESUME_CHECK["passed"] = (RESUME_CHECK["epochs_resumed_B"] == [1, 2]
                              and RESUME_CHECK["resumed_message"] is not None
                              and param_diff <= RESUME_CHECK["param_tolerance"])
    D.write_json(RESUME_CHECK, RUN_DIR / "resume_check.json")
    log("Resume check:", RESUME_CHECK)
    del model_a, model_c
    shutil.rmtree(rc_root, ignore_errors=True)
    if PERSIST_TO_DRIVE:
        shutil.rmtree(PERSIST_RUN_DIR / "resume_check", ignore_errors=True)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    assert RESUME_CHECK["passed"], "resume check failed; see resume_check.json"
else:
    print("Resume check skipped.")

In [ ]:
# Train the main model (or load it). Recovery checkpoints are copied to Drive each epoch.
def copy_tree_atomic(source, destination, last=("COMPLETE.json",)):
    # Copy file by file via a temporary name + rename, so an interrupted copy never
    # leaves a truncated checkpoint under its final name on Drive. Completion
    # markers are copied after every other file.
    files = sorted(p for p in Path(source).rglob("*") if p.is_file())
    files = [p for p in files if p.name not in last] + [p for p in files if p.name in last]
    for path in files:
        target = Path(destination) / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + ".partial")
        shutil.copy2(path, temporary)
        os.replace(temporary, target)

def persist_training(local_dir, subdir):
    if not (PERSIST_TO_DRIVE and PERSIST_DURING_TRAINING):
        return None
    def callback(epoch, row):
        copy_tree_atomic(local_dir, PERSIST_RUN_DIR / subdir)
    return callback

TRAIN_CONFIG = M.TrainConfig(**dict(TRAIN_HPARAMS, seed=SEED))
if TRAIN_MODEL:
    M.set_seed(SEED)
    model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    main_dir = RUN_DIR / "model_training"
    # Cost of one selection evaluation (initial weights), before committing to training.
    t_probe = time.time()
    make_selector(MAIN_KEY)(model)
    SELECTION_SECONDS_ESTIMATE = time.time() - t_probe
    log(f"One selection evaluation ({len(SELECTION_SAMPLE)} requests, full pools + diagnostic AUC): "
        f"{SELECTION_SECONDS_ESTIMATE:.0f}s; per run <= {TRAIN_HPARAMS['max_epochs']} x that"
        + (f"; Optuna adds <= {OPTUNA_N_TRIALS * OPTUNA_EPOCHS} x that" if RUN_OPTUNA else ""))
    M.set_seed(SEED)
    model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    t0 = time.time()
    HISTORY, BEST = M.train_model(model, TENSORS[MAIN_KEY]["train"], make_selector(MAIN_KEY),
                                  SELECTION, TRAIN_CONFIG, main_dir, resume=bool(RESUME_RUN_ID),
                                  on_epoch_end=persist_training(main_dir, "model_training"), log=log)
    TRAIN_SECONDS = time.time() - t0
    SELECTION_PROVENANCE = {"selected_by": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
                            "in_this_run": True}
    log(f"Selected epoch {BEST['epoch']} ({SELECTION_OBJECTIVE} {BEST['key'][0]:.4f}, "
        f"{SELECTION_TIEBREAK} {BEST['key'][1]:.4f} on the selection sample); "
        f"training {TRAIN_SECONDS / 60:.1f} min")
else:
    MAIN_CONFIG = M.ModelConfig.from_json(json.loads((LOAD_DIR / "model_config.json").read_text()))
    model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    model.load_state_dict(torch.load(LOAD_DIR / "model_state.pt", map_location=DEVICE,
                                     weights_only=True))
    source_metrics = json.loads((PERSIST_ROOT / LOAD_RUN_ID / "metrics.json").read_text())
    HISTORY = json.loads((PERSIST_ROOT / LOAD_RUN_ID / "training_history.json").read_text())
    source_selection = source_metrics.get("selection", {})
    BEST = {"epoch": source_selection.get("selected_epoch"),
            "key": source_selection.get("selected_key"), "metrics": source_selection.get("selected_metrics")}
    SELECTION_PROVENANCE = {"selected_by": WEIGHTS_SELECTED_UNDER["objective"],
                            "protocol_version": WEIGHTS_SELECTED_UNDER["protocol_version"],
                            "same_protocol_as_this_run": SAME_PROTOCOL,
                            "in_this_run": False, "source_run_id": LOAD_RUN_ID,
                            "weights_trained_in_run": WEIGHTS_SELECTED_UNDER["trained_in_run"]}
    if SELECTION_PROVENANCE["selected_by"] != SELECTION_OBJECTIVE or not SAME_PROTOCOL:
        log(f"WARNING: {LOAD_RUN_ID} was selected by {SELECTION_PROVENANCE['selected_by']} under protocol "
            f"{SELECTION_PROVENANCE['protocol_version']}, not {SELECTION_OBJECTIVE} under this run's protocol. Its outputs are diagnostic; do not describe it as a model "
            "selected for top-100 shortlisting.")
    TRAIN_SECONDS = None
    log("Loaded model from", LOAD_DIR)
model.eval()
pd.DataFrame(HISTORY)

In [ ]:
# Comparison variants: identical data, hyperparameters, budget and seed.
ABLATIONS = {}

def control_complete(folder):
    # A control counts as finished only if its marker exists and both result files
    # match the hashes recorded in it; otherwise it resumes from its recovery
    # checkpoint (if any) or is retrained.
    marker = folder / "COMPLETE.json"
    if not marker.is_file():
        return False
    recorded = json.loads(marker.read_text())["files"]
    return all((folder / name).is_file() and D.sha256_file(folder / name) == digest
               for name, digest in recorded.items())

def save_control_result(folder, summary, per):
    # Write both result files atomically, then the completion marker last.
    tmp = folder / "validation_per_impression.npz.tmp"
    with open(tmp, "wb") as handle:
        np.savez_compressed(handle, **per)
    os.replace(tmp, folder / "validation_per_impression.npz")
    D.write_json(summary, folder / "summary.json")
    np.load(folder / "validation_per_impression.npz")["auc"]       # readable
    json.loads((folder / "summary.json").read_text())              # readable
    D.write_json({"files": {n: D.sha256_file(folder / n) for n in
                            ("summary.json", "validation_per_impression.npz")}},
                 folder / "COMPLETE.json")
    assert control_complete(folder)

if RUN_ABLATIONS:
    if "attention_shuffled" in ABLATION_VARIANTS and TRAIN_CONFIG.history_dropout > 0:
        log("NOTE: history_dropout > 0 drops events in the (shuffled) array order, so the "
            "shuffled control is then not a pure order-only comparison.")
    for variant in ABLATION_VARIANTS:
        if variant == MODEL_VARIANT:
            continue
        ab_dir = RUN_DIR / "ablations" / variant
        if control_complete(ab_dir):   # finished and verified before an interruption
            saved = json.loads((ab_dir / "summary.json").read_text())
            per = dict(np.load(ab_dir / "validation_per_impression.npz"))
            if saved.get("selection") != SELECTION:
                raise RuntimeError(f"{ab_dir} was selected under a different declaration; "
                                   "use a new run for the controls")
            ABLATIONS[variant] = {"history": saved["history"], "best_epoch": saved["best_epoch"],
                                  "metrics": saved["validation"], "per": per}
            log(f"--- ablation {variant}: reusing finished result")
            continue
        key = "shuffled" if variant == "attention_shuffled" else "ordered"
        if key not in TENSORS:
            ARRAYS[key] = make_arrays(variant, ("train", "validation"))
            TENSORS[key] = {s: M.SplitTensors(ARRAYS[key][s], DEVICE) for s in ("train", "validation")}
        if variant == "attention_shuffled":   # order-only: identical event features
            for s_ in ("train", "validation"):
                o, sh = ARRAYS["ordered"][s_], ARRAYS[key][s_]
                assert np.allclose(np.sort(o.hist_features.sum(-1), 1), np.sort(sh.hist_features.sum(-1), 1))
                assert np.array_equal(np.sort(o.hist_times_s, 1), np.sort(sh.hist_times_s, 1))
        M.set_seed(SEED)
        ab_model = M.build_model(model_config(variant), ARTICLE_TENSORS, DEVICE)
        log(f"--- ablation: {variant}")
        hist, best = M.train_model(ab_model, TENSORS[key]["train"], make_selector(key), SELECTION,
                                   TRAIN_CONFIG, ab_dir,
                                   resume=bool(RESUME_RUN_ID),
                                   on_epoch_end=persist_training(ab_dir, f"ablations/{variant}"),
                                   log=log)
        evaluated = make_selector(key)(ab_model, return_per_request=True)
        per = {"recall": evaluated["_per_request"]["recall"], "ndcg": evaluated["_per_request"]["ndcg"],
               "auc": evaluated["_slate"]["_per_impression"]["auc"]}
        metrics = {k: v for k, v in evaluated.items() if not k.startswith("_")}
        ABLATIONS[variant] = {"history": hist, "best_epoch": best["epoch"], "metrics": metrics,
                              "per": per}
        save_control_result(ab_dir, {"variant": variant, "best_epoch": best["epoch"], "history": hist,
                                     "selection": SELECTION, "validation": metrics}, per)
        if PERSIST_TO_DRIVE:   # results first, marker last; then verify the Drive copy
            drive_dir = PERSIST_RUN_DIR / "ablations" / variant
            copy_tree_atomic(ab_dir, drive_dir)
            assert control_complete(drive_dir), f"{variant}: Drive copy incomplete"
        # Checkpoints are deleted only after the complete result is verified.
        (ab_dir / "best_model_state.pt").unlink(missing_ok=True)   # metrics only; keep bundle small
        shutil.rmtree(ab_dir / "recovery", ignore_errors=True)
        if PERSIST_TO_DRIVE:
            (drive_dir / "best_model_state.pt").unlink(missing_ok=True)
            shutil.rmtree(drive_dir / "recovery", ignore_errors=True)
        del ab_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("Ablations skipped.")

## 6. Validation diagnostics and explainability

All numbers below are **development validation** results. Validation is used for
early stopping, selection and any tuning, so it is not an untouched test. Section 6
covers:
1. the selected model on the fixed selection sample: catalogue known-click
   Recall@100 (primary) and observed-click nDCG@100 (secondary), each over the full
   eligible pool, plus the recorded-slate AUC as a diagnostic. The full validation
   catalogue is scored in Section 7;
2. the two focused comparisons, on the same requests and pools, with paired
   user-cluster bootstrap intervals for Recall@100 (AUC differences are diagnostic);
3. what the attention looks at: recency profile and same-category lift (a
   diagnostic, not a causal explanation);
4. performance by history length and for clicks on articles never referenced in
   training (cold content).

In [ ]:
SELECTED_EVAL = make_selector(MAIN_KEY)(model, return_per_request=True)
SELECTED_PER = SELECTED_EVAL.pop("_per_request")
SLATE = SELECTED_EVAL.pop("_slate")
per_imp = SLATE.pop("_per_impression")            # recorded-slate diagnostic per impression
VAL_METRICS = {"selection_sample_catalogue": SELECTED_EVAL,
               "diagnostic_recorded_slate": SLATE}
log("Selection sample, selected model:", SELECTED_EVAL)
log("Diagnostic recorded-slate metrics:", SLATE)

# Comparisons on identical requests and full pools; paired user-cluster bootstrap.
SAMPLE_USERS = VAL.user_id[SELECTION_SAMPLE]
RESTORE_CHECK = None
if TRAIN_MODEL and BEST.get("key"):   # the restored checkpoint should reproduce its key
    RESTORE_CHECK = {"selected_key": BEST["key"],
                     "recomputed_key": [SELECTED_EVAL[SELECTION_OBJECTIVE], SELECTED_EVAL[SELECTION_TIEBREAK]]}
    RESTORE_CHECK["matches"] = bool(np.allclose(RESTORE_CHECK["selected_key"],
                                                RESTORE_CHECK["recomputed_key"], atol=1e-9))
    log("Selected-checkpoint restore check:", RESTORE_CHECK)
    if not RESTORE_CHECK["matches"]:
        log("WARNING: recomputed selection metrics differ from the selected checkpoint's "
            "(possible GPU near-tie nondeterminism at the cut-off, or a restore problem); "
            "inspect before relying on the run.")
label = (f"{MODEL_VARIANT} (selected by {SELECTION_PROVENANCE['selected_by']})")
rows = [{"variant": label, "best_epoch": BEST["epoch"],
         "recall@100": SELECTED_EVAL[SELECTION_OBJECTIVE],
         "observed_ndcg@100": SELECTED_EVAL[SELECTION_TIEBREAK],
         "diag_auc": SLATE["mean_impression_auc"]}]
COMPARISONS = {}
for variant, result in ABLATIONS.items():
    other = result["per"]
    boot = D.user_cluster_bootstrap(SAMPLE_USERS, SELECTED_PER["recall"], other["recall"],
                                    BOOTSTRAP_SAMPLES, SEED)
    boot_auc = D.user_cluster_bootstrap(SAMPLE_USERS, per_imp["auc"], other["auc"],
                                        BOOTSTRAP_SAMPLES, SEED)
    COMPARISONS[variant] = {"recall@100_difference_selected_minus_variant": boot,
                            "diagnostic_auc_difference_selected_minus_variant": boot_auc}
    rows.append({"variant": variant, "best_epoch": result["best_epoch"],
                 "recall@100": result["metrics"][SELECTION_OBJECTIVE],
                 "observed_ndcg@100": result["metrics"][SELECTION_TIEBREAK],
                 "diag_auc": result["metrics"]["diagnostic_recorded_slate_auc"],
                 "delta_recall_selected_minus_variant": boot["mean_difference"],
                 "ci95_low": boot["ci95"][0], "ci95_high": boot["ci95"][1]})
COMPARISON_TABLE = pd.DataFrame(rows)
if SELECTION_PROVENANCE["selected_by"] != SELECTION_OBJECTIVE and ABLATIONS:
    log("WARNING: mixed selection provenance - the main model was selected by "
        f"{SELECTION_PROVENANCE['selected_by']}, the controls by {SELECTION_OBJECTIVE}.")

# Learning curves: training loss and the selection metrics per evaluation.
import matplotlib.pyplot as plt
curve = pd.DataFrame(HISTORY)
if len(curve) and SELECTION_OBJECTIVE in curve:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(curve["epoch"], curve["train_bce"], marker="o"); axes[0].set_title("train BCE (observed pairs)")
    for name in (SELECTION_OBJECTIVE, SELECTION_TIEBREAK, "diagnostic_recorded_slate_auc"):
        if name in curve:
            axes[1].plot(curve["epoch"], curve[name], marker="o", label=name)
    if BEST.get("epoch"):
        axes[1].axvline(BEST["epoch"], ls="--", color="grey", label="selected")
    axes[1].set_title("validation (selection sample)"); axes[1].legend(fontsize=8)
    for ax in axes:
        ax.set_xlabel("epoch")
    fig.tight_layout(); fig.savefig(RUN_DIR / "learning_curves.png", dpi=120); plt.show()
COMPARISON_TABLE

In [ ]:
# Recall@100 by history length (selection sample, full pools) and diagnostic AUC
# for "cold" clicked articles (never referenced in train).
lengths = VAL.hist_full_lengths[VAL.imp_user_row[SELECTION_SAMPLE]]
bins = [-1, 0, 9, 49, 199, np.inf]
labels_ = ["0", "1-9", "10-49", "50-199", "200+"]
bucket = pd.cut(lengths, bins=bins, labels=labels_)
by_length = (pd.DataFrame({"history_length": bucket, "recall": SELECTED_PER["recall"],
                           "ndcg": SELECTED_PER["ndcg"]})
             .groupby("history_length", observed=False)
             .agg(requests=("recall", "size"), with_known_clicks=("recall", "count"),
                  recall_at_100=("recall", "mean"), observed_ndcg_at_100=("ndcg", "mean")))
sizes = np.diff(VAL.offsets)
imp_of = np.repeat(np.arange(VAL.n_impressions), sizes)
clicked = VAL.labels == 1
cold = ~np.isin(VAL.cand_article_id, np.fromiter(seen_train_ids, dtype=np.int64))
cold_click_imp = np.zeros(VAL.n_impressions, bool)
np.logical_or.at(cold_click_imp, imp_of[clicked & cold], True)
sample_cold = cold_click_imp[SELECTION_SAMPLE]
by_cold = (pd.DataFrame({"clicked_article_unseen_in_train": sample_cold,
                         "recall": SELECTED_PER["recall"]})
           .groupby("clicked_article_unseen_in_train")
           .agg(requests=("recall", "size"), recall_at_100=("recall", "mean")))
DIAGNOSTICS = {"by_history_length": by_length.reset_index().to_dict("records"),
               "by_cold_click": by_cold.reset_index().to_dict("records"),
               "validation_candidates_unseen_in_train": float(cold.mean())}
display(by_length)
display(by_cold)

In [ ]:
# Attention diagnostics on a fixed sample of validation impressions.
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
sample_imps = np.sort(rng.choice(VAL.n_impressions, size=min(2000, VAL.n_impressions), replace=False))
records = M.collect_attention(model, TENSORS[MAIN_KEY]["validation"], sample_imps)
if MODEL_VARIANT == "attention_shuffled":
    print("Main model uses shuffled histories: the recency-rank plot below is not meaningful.")
ATTENTION_DIAG = None
if records:
    max_r = 20
    weight_by_recency, count_by_recency = np.zeros(max_r), np.zeros(max_r)
    uniform_by_recency = np.zeros(max_r)
    same_weight = same_uniform = 0.0
    for rec in records:
        w, L = rec["weights"], rec["hist_len"]
        cand_cat = ARTICLES.category[VAL.cand_article_row[rec["cand_index"]]]
        hist_cat = ARTICLES.category[rec["hist_rows"]]
        for r in range(max_r):
            pos = L - 1 - r
            ok = pos >= 0
            weight_by_recency[r] += w[np.flatnonzero(ok), pos[ok]].sum()
            uniform_by_recency[r] += (1.0 / L[ok]).sum()
            count_by_recency[r] += ok.sum()
        valid = np.arange(w.shape[1])[None, :] < L[:, None]
        match = valid & (hist_cat == cand_cat[:, None]) & (cand_cat[:, None] >= 3)
        has = L > 0
        same_weight += (w * match)[has].sum()
        same_uniform += (match.sum(1)[has] / L[has]).sum()
    ATTENTION_DIAG = {
        "impressions_sampled": int(len(sample_imps)),
        "mean_weight_by_recency_rank": (weight_by_recency / np.maximum(count_by_recency, 1)).tolist(),
        "uniform_weight_by_recency_rank": (uniform_by_recency / np.maximum(count_by_recency, 1)).tolist(),
        "same_category_attention_lift": float(same_weight / same_uniform) if same_uniform else None,
    }
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(range(1, max_r + 1), ATTENTION_DIAG["mean_weight_by_recency_rank"], marker="o",
            label="mean attention weight")
    ax.plot(range(1, max_r + 1), ATTENTION_DIAG["uniform_weight_by_recency_rank"], ls="--",
            label="uniform over history")
    ax.set_xlabel("history event recency rank (1 = most recent)")
    ax.set_ylabel("weight"); ax.legend(); ax.set_title("Validation attention by recency (diagnostic)")
    fig.tight_layout(); fig.savefig(RUN_DIR / "attention_recency.png", dpi=120); plt.show()
    log("Same-category attention lift (1 = no preference):", ATTENTION_DIAG["same_category_attention_lift"])
else:
    print("Selected variant has no attention weights.")

In [ ]:
# One explained ranking (diagnostic): top candidates and the history they attend to.
if records:
    titles = pd.concat([pd.read_parquet(p, columns=["article_id", "title", "category_str"])
                        for p in (PATHS["articles_train_val"], PATHS["articles_test"])]
                       ).drop_duplicates("article_id").set_index("article_id")
    rec = records[0]
    first_imp = sample_imps[0]
    lo, hi = VAL.offsets[first_imp], VAL.offsets[first_imp + 1]
    local = np.flatnonzero((rec["cand_index"] >= lo) & (rec["cand_index"] < hi))
    order = local[np.argsort(-rec["logits"][local])]
    rows = []
    for j in order[:5]:
        aid = int(VAL.cand_article_id[rec["cand_index"][j]])
        top = np.argsort(-rec["weights"][j])[:3]
        attended = [ARTICLES.article_ids[rec["hist_rows"][j][t]] for t in top if t < rec["hist_len"][j]]
        rows.append({"article_id": aid, "title": titles["title"].get(aid),
                     "category": titles["category_str"].get(aid),
                     "score": float(rec["logits"][j]),
                     "clicked": int(VAL.labels[rec["cand_index"][j]]),
                     "top_attended_history": [titles["title"].get(int(a)) for a in attended]})
    print(f"Validation behaviour {VAL.user_id[first_imp]}:{VAL.impression_id[first_imp]} "
          f"(history length {VAL.hist_lengths[VAL.imp_user_row[first_imp]]})")
    with pd.option_context("display.max_colwidth", 80):
        display(pd.DataFrame(rows))

**Interpretation (fill in after running).**
- **Attention vs `last_state`:** Δ Recall@100 = … (95% CI …); diagnostic Δ AUC = ….
  Decision: keep target-aware attention only if the Recall@100 CI excludes 0
  **and** the same-category lift is clearly > 1. Otherwise the simpler summary is
  sufficient.
- **Ordered vs shuffled (order only):** Δ Recall@100 = … (95% CI …). Both variants see
  identical per-event features, so if the CI includes 0, the LSTM's use of order
  adds nothing beyond those time features, and a pooled model may be enough.
- **History length / cold clicks:** …
- **Adjustments made:** … (record any configuration change and why; do not iterate
  on validation for marginal gains).

## 7. Predict and export all splits

Section 7 runs in four steps:
1. **Save the model bundle** and persist it, before the long catalogue scoring.
2. **Recorded-slate diagnostics:** score every recorded candidate once and write
   `diagnostics/recorded_slates/{split}_scores.csv`. This is the labelled
   benchmark, not a catalogue submission.
3. **Feasibility estimate:** count eligible pairs exactly; measure the score-CSV
   row size, the CSV write/read speed, the Drive copy speed and the scoring speed
   on sample requests; and compare the cumulative export size with free disk
   space. Full catalogue score files are written only when they fit
   (`FULL_SCORE_EXPORT`). Otherwise the scores of a declared, seeded request
   sample are written to `predictions/{split}/sampled_requests/`, and the run
   records that the full export is incomplete. If the estimated time exceeds
   `MAX_CATALOGUE_HOURS`, no split is scored in this session. The run is still
   recorded, and the split can be scored later with `RESUME_RUN_ID` and a smaller
   `CATALOGUE_SPLITS`.
4. **Catalogue scoring:** for every export request of the run scope (all of them
   in a `full` run, otherwise the shared hash sample, written under
   `sampled_requests/`), score all eligible union-catalogue articles and
   write the exact top-100 to `rankings/{split}/submission_top_100.csv`. Scores
   are sorted by descending score, ties by ascending article ID, keeping
   `min(100, pool)` unique IDs and writing `[]` for an empty pool. A per-request
   `eligibility_manifest.csv` records pool sizes. Before scoring a split, the
   catalogue path is checked against the recorded-slate path. The written files
   are re-read and checked; for requests with exported scores, the shortlist is
   recomputed from the score rows. A split is finished only once
   `catalogue/{split}.json` has been written, after its files. A resumed run reuses
   finished splits; an unfinished split restarts from its beginning.

Score semantics:
- `score` is the fp32 logit.
- `click_probability = sigmoid(score)` is uncalibrated.
- Outside the recorded slate, outcomes are unknown, so no catalogue AUC/BCE is
  computed.
- Train outputs are in-sample.

In [ ]:
# 7.1 Model bundle (saved and persisted before catalogue scoring).
MODEL_DIR = RUN_DIR / "model"
(MODEL_DIR / "code").mkdir(parents=True, exist_ok=True)
M.atomic_torch_save(model.state_dict(), MODEL_DIR / "model_state.pt")
D.write_json(MAIN_CONFIG.to_json(), MODEL_DIR / "model_config.json")
D.write_json(PREPROCESSING, MODEL_DIR / "preprocessing.json")
D.write_json({"policy": POLICY.to_json(), "shortlist_size": SHORTLIST_SIZE,
              "final_display_size": FINAL_DISPLAY_SIZE}, MODEL_DIR / "catalogue_policy.json")
np.savez_compressed(
    MODEL_DIR / "mappings.npz",
    article_table_ids=ARTICLES.article_ids,
    id_vocab_ids=np.array(list(ID_VOCAB.keys()), dtype=np.int64),
    id_vocab_rows=np.array(list(ID_VOCAB.values()), dtype=np.int64),
    train_referenced_ids=np.array(sorted(seen_train_ids), dtype=np.int64))
for name in ("lstm_model.py", "lstm_data.py"):
    shutil.copy2(CODE_DIR / name, MODEL_DIR / "code" / name)
if TRAIN_MODEL:
    (RUN_DIR / "training_history.json").write_text(json.dumps(HISTORY, indent=2))
    pd.DataFrame(HISTORY).to_csv(RUN_DIR / "training_history.csv", index=False)
    recovery_src = RUN_DIR / "model_training" / "recovery" / "last_checkpoint.pt"
    if recovery_src.is_file():
        (MODEL_DIR / "recovery").mkdir(exist_ok=True)
        shutil.copy2(recovery_src, MODEL_DIR / "recovery" / "last_checkpoint.pt")
else:
    shutil.copy2(PERSIST_ROOT / LOAD_RUN_ID / "training_history.json", RUN_DIR / "training_history.json")

requirements = [f"torch=={torch.__version__}", f"numpy=={np.__version__}",
                f"pandas=={pd.__version__}", f"pyarrow=={pa.__version__}"]
for dist in ("matplotlib", "duckdb"):
    if installed(dist):
        requirements.append(f"{dist}=={installed(dist)}")
if TUNING_SUMMARY:
    requirements.append(f"optuna=={TUNING_SUMMARY['optuna_version']}")
(RUN_DIR / "requirements.txt").write_text("\n".join(requirements) + "\n")
if PERSIST_TO_DRIVE:
    copy_tree_atomic(MODEL_DIR, PERSIST_RUN_DIR / "model")
    for name in ("training_history.json", "training_history.csv", "requirements.txt"):
        if (RUN_DIR / name).is_file():
            shutil.copy2(RUN_DIR / name, PERSIST_RUN_DIR / name)
log("Model bundle saved:", MODEL_DIR)

In [ ]:
# 7.2 Recorded-slate diagnostics (labelled benchmark rows; not catalogue submissions).
# Scope: every recorded candidate of the split's EXPORT requests (all requests in a
# full run; the declared sample otherwise, written under sampled_requests/).
DIAG_DIR = RUN_DIR / "diagnostics" / "recorded_slates"
DIAG_INFO, DIAG_PATHS = {}, {}
for split in D.SPLITS:
    arrays = D.subset_split_arrays(ARRAYS[MAIN_KEY][split], REQUESTS[split])
    logits = M.predict_logits(model, M.SplitTensors(arrays, DEVICE))
    assert np.isfinite(logits).all(), split
    probs = 1.0 / (1.0 + np.exp(-logits)) if EXPORT_CLICK_PROBABILITY else None
    scores = D.score_frame(arrays, logits, probs)
    DIAG_INFO[split] = {**D.validate_exports(arrays, scores, D.ranking_frame(scores)),
                        "requests": int(arrays.n_impressions), "sampled_requests": LIMITED[split],
                        "scope": "recorded slates of the export requests",
                        "request_ids_sha256": REQUEST_SCOPE["export"][split]["ids_sha256"]}
    DIAG_PATHS[split] = (DIAG_DIR / "sampled_requests" if LIMITED[split] else DIAG_DIR) / f"{split}_scores.csv"
    D.write_csv_atomic(scores, DIAG_PATHS[split])
    if split == "validation":
        check = pd.read_csv(DIAG_PATHS[split], dtype={"behaviour_id": str, "article_id": "int64"},
                            float_precision="round_trip")
        assert np.array_equal(check["score"].to_numpy(), scores["score"].to_numpy())
    log(split, "recorded-slate diagnostics:", DIAG_INFO[split])
if PERSIST_TO_DRIVE:
    copy_tree_atomic(DIAG_DIR, PERSIST_RUN_DIR / "diagnostics" / "recorded_slates")

In [ ]:
# 7.3 Feasibility: exact eligible pairs, measured row size, scoring and I/O speed,
# cumulative free space. Splits already finished in this run folder are skipped.
import io
rng_cat = np.random.default_rng(SEED)
CATALOGUE_DIR = RUN_DIR / "catalogue"
CATALOGUE_DIR.mkdir(exist_ok=True)

# Requests scored per split were declared in Section 4 (REQUESTS / LIMITED): all
# requests, or the shared hash sample; each always uses its FULL eligible pool.
def split_record(split):
    # A split is finished only if its record exists, matches this request policy and
    # every file it names exists.
    path = CATALOGUE_DIR / f"{split}.json"
    if not path.is_file():
        return None
    record = json.loads(path.read_text())
    if (record.get("protocol_sha256") != SELECTION["protocol_sha256"]
            or record.get("request_ids_sha256") != REQUEST_SCOPE["export"][split]["ids_sha256"]):
        return None
    files = [record["files"][k] for k in ("rankings", "manifest", "scores") if record["files"].get(k)]
    return record if all((RUN_DIR / f).is_file() for f in files) else None

scorers = {}
def get_scorer(split):
    if split not in scorers:
        scorers[split] = M.CatalogueScorer(model, TENSORS[MAIN_KEY][split], CAT_INDEX[split],
                                           PREPROCESSING["numeric_stats"]["log_candidate_age_h"],
                                           MAX_ARTICLE_AGE_HOURS, CATALOGUE_CHUNK,
                                           requests=REQUESTS[split])
    return scorers[split]

pending = [s for s in CATALOGUE_SPLITS if split_record(s) is None]
done = [s for s in CATALOGUE_SPLITS if s not in pending]
if done:
    log("Catalogue splits already finished in this run (reused):", done)

# Measured CSV row size and write/read speed, using a real request's scores.
probe_split = pending[0] if pending else CATALOGUE_SPLITS[0]
probe = get_scorer(probe_split)
probe_reqs = REQUESTS[probe_split]
probe_ids, probe_logits = probe.score_request(
    int(probe_reqs[np.argmax(probe.hi[probe_reqs] - probe.lo[probe_reqs])]))
probe_frame = pd.DataFrame({"behaviour_id": "9999999:9999999999",
                            "article_id": probe_ids.cpu().numpy(),
                            "score": probe_logits.cpu().numpy().astype(np.float64)})
probe_frame["click_probability"] = 1 / (1 + np.exp(-probe_frame["score"]))
probe_path = LOCAL_ROOT / "_io_probe.csv"
t0 = time.time(); probe_frame.to_csv(probe_path, index=False); write_s = time.time() - t0
t0 = time.time(); pd.read_csv(probe_path, float_precision="round_trip"); read_s = time.time() - t0
probe_bytes = probe_path.stat().st_size
copy_mb_s = None
if PERSIST_TO_DRIVE and probe_bytes:
    t0 = time.time(); shutil.copy2(probe_path, PERSIST_ROOT / "_io_probe.csv")
    copy_mb_s = probe_bytes / 1e6 / max(time.time() - t0, 1e-3)
    (PERSIST_ROOT / "_io_probe.csv").unlink(missing_ok=True)
probe_path.unlink(missing_ok=True)
rows = max(len(probe_frame), 1)
ROW_BYTES = probe_bytes / rows if len(probe_frame) else 64.0
WRITE_ROWS_S = rows / max(write_s, 1e-3) if len(probe_frame) else 1.5e5
READ_ROWS_S = rows / max(read_s, 1e-3) if len(probe_frame) else 4e5

def free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except OSError:
        return None

FEASIBILITY = {"row_bytes_measured": ROW_BYTES, "csv_write_rows_per_s": WRITE_ROWS_S,
               "csv_read_rows_per_s": READ_ROWS_S, "drive_copy_mb_per_s": copy_mb_s,
               "local_free_gb": free_gb(LOCAL_ROOT),
               "persistent_free_gb": free_gb(PERSIST_ROOT) if PERSIST_TO_DRIVE else None,
               "splits": {}, "already_finished": done}
capacity = [v for v in (FEASIBILITY["local_free_gb"], FEASIBILITY["persistent_free_gb"]) if v is not None]
budget_gb = min(capacity) / 1.25 if capacity else 0.0     # cumulative across splits
total_hours = 0.0
for split in pending:
    scorer = get_scorer(split)
    summary = D.eligibility_summary(ARRAYS[MAIN_KEY][split], CAT_INDEX[split],
                                    MAX_ARTICLE_AGE_HOURS, ROW_BYTES, requests=REQUESTS[split])
    summary["sampled_requests"] = LIMITED[split]
    timed = rng_cat.choice(REQUESTS[split], size=min(30, len(REQUESTS[split])), replace=False)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for i in timed:
        scorer.shortlist(int(i), SHORTLIST_SIZE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    per_request = (time.time() - t0) / len(timed)
    score_hours = summary["requests"] * per_request / 3600 * 1.15      # +15% bookkeeping
    gb = summary["estimated_full_score_csv_gb"]
    pairs = summary["eligible_pairs"]
    io_hours = (pairs / WRITE_ROWS_S + pairs / READ_ROWS_S
                + (gb * 1e3 / copy_mb_s if copy_mb_s else 0.0)) / 3600
    fits = gb <= budget_gb
    if FULL_SCORE_EXPORT is True:
        if not fits:
            raise RuntimeError(f"{split}: full score export needs ~{gb:.1f} GB; "
                               f"remaining budget {budget_gb:.1f} GB")
        full = True
    elif FULL_SCORE_EXPORT == "auto":
        full = fits and gb <= FULL_SCORE_EXPORT_MAX_GB
    else:
        full = False
    if full:
        budget_gb -= gb
    hours = score_hours + (io_hours if full else 0.0)
    FEASIBILITY["splits"][split] = {
        **summary, "timed_requests": int(len(timed)), "seconds_per_request": per_request,
        "scoring_hours": score_hours, "full_export_io_hours": io_hours, "estimated_hours": hours,
        "full_score_export": full,
        "full_score_export_reason": ("fits the size cap and remaining free space" if full else
                                     f"~{gb:.1f} GB vs cap {FULL_SCORE_EXPORT_MAX_GB} GB / "
                                     f"remaining space {budget_gb:.1f} GB" if FULL_SCORE_EXPORT
                                     else "FULL_SCORE_EXPORT=False")}
    total_hours += hours
FEASIBILITY["estimated_total_hours"] = total_hours
# Cost of scoring EVERY request of each split against the union pool (planning only).
def population_cost(split):
    times = (TRAIN_POPULATION["time_us"] if split == "train" else ARRAYS[MAIN_KEY][split].imp_time_us)
    lo, hi = CAT_INDEX[split].bounds(times)
    pairs = int((hi - lo).sum())
    return {"requests": int(len(times)), "eligible_pairs": pairs,
            "empty_pools": int(((hi - lo) == 0).sum()),
            "estimated_full_score_csv_gb": pairs * ROW_BYTES / 1e9}
FEASIBILITY["all_requests_cost"] = {s: population_cost(s) for s in D.SPLITS}
FEASIBILITY["all_requests_cost_note"] = (
    "every request x union pool; GB = score CSV estimate at the measured row size")
CATALOGUE_SKIPPED = None
if total_hours > MAX_CATALOGUE_HOURS:
    CATALOGUE_SKIPPED = (f"estimated {total_hours:.1f} h > MAX_CATALOGUE_HOURS={MAX_CATALOGUE_HOURS}; "
                         "no catalogue split scored in this session")
FEASIBILITY["skipped"] = CATALOGUE_SKIPPED
D.write_json(FEASIBILITY, RUN_DIR / "catalogue_feasibility.json")
if FEASIBILITY["splits"]:
    display(pd.DataFrame(FEASIBILITY["splits"]).T[[
        "requests", "requests_in_split", "sampled_requests", "eligible_pairs", "empty_pools",
        "pool_size_min_median_max",
        "pair_share_with_text_vector", "estimated_full_score_csv_gb", "scoring_hours",
        "full_export_io_hours", "full_score_export"]])
log(f"Row bytes {ROW_BYTES:.1f}; CSV write {WRITE_ROWS_S:,.0f} rows/s; Drive copy "
    f"{copy_mb_s} MB/s; free GB {capacity}; estimated {total_hours:.2f} h")
if PERSIST_TO_DRIVE:
    shutil.copy2(RUN_DIR / "catalogue_feasibility.json", PERSIST_RUN_DIR / "catalogue_feasibility.json")
if CATALOGUE_SKIPPED:
    log("WARNING:", CATALOGUE_SKIPPED, "- raise MAX_CATALOGUE_HOURS, or set CATALOGUE_SPLITS to "
        "fewer splits and continue this run later with RESUME_RUN_ID. Metrics, run config and "
        "persistence below still run and record the catalogue outputs as missing.")

In [ ]:
# 7.4 Catalogue scoring: exact top-100 for every request, streamed to disk.
# Each finished split writes catalogue/{split}.json last, so a later session with
# RESUME_RUN_ID reuses finished splits instead of rescoring them.
to_score = [] if CATALOGUE_SKIPPED else pending
for split in to_score:
    arrays, scorer = ARRAYS[MAIN_KEY][split], get_scorer(split)
    requests = REQUESTS[split]
    n = len(requests)
    # Catalogue path must reproduce the recorded-slate path: check before scoring.
    checked = D.first_by_priority(requests, EXPORT_BIDS[split], split, SAMPLING_SEED,
                                  min(CONSISTENCY_CHECK_REQUESTS, n))
    worst, compared = 0.0, 0
    slate_logits = M.predict_logits(model, TENSORS[MAIN_KEY][split], impression_subset=checked)
    for i in checked:
        ids, logits = scorer.score_request(int(i))
        position = {a: j for j, a in enumerate(ids.cpu().tolist())}
        lg = logits.cpu().numpy()
        for r in range(arrays.offsets[i], arrays.offsets[i + 1]):
            j = position.get(int(arrays.cand_article_id[r]))
            if j is not None:
                worst = max(worst, abs(float(lg[j]) - slate_logits[r])); compared += 1
    consistency = {"requests": int(len(checked)), "recorded_candidates_compared": compared,
                   "max_abs_logit_diff": worst}
    assert worst <= RELOAD_TOLERANCE, f"{split}: catalogue and recorded-slate scores differ ({worst})"
    log(split, "catalogue/recorded-slate consistency:", consistency)

    full = FEASIBILITY["splits"][split]["full_score_export"]
    sampled = set() if full else set(D.first_by_priority(
        requests, EXPORT_BIDS[split], split, SAMPLING_SEED, min(SAMPLED_SCORE_REQUESTS, n)).tolist())
    writer = D.ShortlistWriter(RUN_DIR, split, SHORTLIST_SIZE, full_scores=full,
                               sampled_requests=LIMITED[split])
    bids = D.behaviour_ids(arrays.user_id, arrays.impression_id)
    times = pd.to_datetime(arrays.imp_time_us, unit="us").map(lambda t: t.isoformat()).to_numpy()
    click_lookup = D.clicks_by_request(CLICK_ELIGIBILITY[split])
    hits, lengths, text_slots, ties = {}, np.zeros(n, np.int64), 0, 0
    t0 = time.time()
    for position, i in enumerate(requests.tolist()):
        top_ids, _, ids, logits = scorer.shortlist(i, SHORTLIST_SIZE)
        ties += int(scorer.last_tie_at_cutoff)
        top = top_ids.cpu().numpy()
        export = full or i in sampled
        writer.add(bids[i], times[i], len(ids), top,
                   logits.cpu().numpy() if export else None, ids.cpu().numpy() if export else None)
        lengths[position] = len(top)
        text_slots += int(ARTICLES.has_text[ARTICLES.rows_for(top)].sum())
        h = D.click_hits(click_lookup, i, top, (SHORTLIST_SIZE,))
        if h is not None:
            hits[i] = h
        if (position + 1) % 20000 == 0:
            rate = (position + 1) / (time.time() - t0)
            log(f"  {split}: {position + 1:,}/{n:,} requests, {rate:.0f}/s, "
                f"ETA {(n - position - 1) / rate / 60:.1f} min")
    outputs = writer.close()
    validation = D.validate_catalogue_exports(arrays, CAT_INDEX[split], outputs, SHORTLIST_SIZE,
                                              MAX_ARTICLE_AGE_HOURS, requests=requests)
    relative = lambda p: str(Path(p).relative_to(RUN_DIR)) if p else None
    record = {
        "split": split, "files": {"rankings": relative(outputs["rankings"]),
                                  "manifest": relative(outputs["manifest"]),
                                  "scores": relative(outputs["scores"])},
        "protocol_sha256": SELECTION["protocol_sha256"],
        "request_ids_sha256": REQUEST_SCOPE["export"][split]["ids_sha256"],
        "request_ids_file": REQUEST_SCOPE["export"][split]["file"],
        "export_requests_limit": EXPORT_REQUESTS, "sampled_requests": LIMITED[split],
        "requests_scored": n, "requests_in_split": population_size(split),
        "full_score_export": full,
        "score_requests_exported": "all scored requests" if full else len(sampled),
        "score_rows": outputs["score_rows"], "validation": validation, "consistency": consistency,
        "metrics": {"requests": n, "eligible_pairs": validation["eligible_pairs"],
                    "empty_pools": validation["empty_pools"], "seconds": time.time() - t0,
                    "shortlist_length_min_median_max": [int(lengths.min()), float(np.median(lengths)),
                                                        int(lengths.max())] if n else None,
                    "shortlist_text_vector_share": text_slots / max(int(lengths.sum()), 1),
                    "in_sample": split == "train", "sampled_requests": LIMITED[split],
                    "scope": "exported requests (not the selection sample)",
                    "ties_at_cutoff": ties,
                    **{k: v for k, v in D.recall_from_hits(
                        hits, CLICK_ELIGIBILITY[split], (SHORTLIST_SIZE,), requests_scored=n).items()
                       if k != "split_click_eligibility"}}}
    if PERSIST_TO_DRIVE:   # data files first; the split record is published last
        for sub in ("rankings", "predictions"):
            copy_tree_atomic(RUN_DIR / sub / split, PERSIST_RUN_DIR / sub / split)
    D.write_json(record, CATALOGUE_DIR / f"{split}.json")
    if PERSIST_TO_DRIVE:
        copy_tree_atomic(CATALOGUE_DIR, PERSIST_RUN_DIR / "catalogue")
    log(split, "catalogue:", record["metrics"])
    del hits, click_lookup
    scorers.pop(split, None)          # free this split's encodings/states before the next
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
del scorers
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
CATALOGUE_RECORDS = {s: split_record(s) for s in D.SPLITS if split_record(s)}
log("Catalogue splits finished in this run:", sorted(CATALOGUE_RECORDS))

In [ ]:
# 7.5 Metrics, run configuration and export manifest.
METRICS = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID,
    "partition": "validation (development: used for early stopping/selection"
                 + ("/tuning" if TUNING_SUMMARY else "") + "; not an untouched test)",
    "primary_metric": f"validation {SELECTION_OBJECTIVE} (catalogue, full eligible pool per request)",
    # Two DIFFERENT validation request sets; compare runs only on identical request IDs/pools.
    "selection_validation": {
        "scope": "selection sample at the selected checkpoint (used for selection)",
        "requests": REQUEST_SCOPE["selection"], **VAL_METRICS["selection_sample_catalogue"]},
    "exported_validation": {
        "scope": "exported validation requests (scored after selection)",
        "request_ids": REQUEST_SCOPE["export"]["validation"],
        "metrics": (CATALOGUE_RECORDS.get("validation") or {}).get("metrics")},
    "exported_validation_subset_of_selection": REQUEST_SCOPE["exported_validation_subset_of_selection"],
    "selected_checkpoint_restore_check": RESTORE_CHECK,
    "diagnostic_recorded_slate": VAL_METRICS["diagnostic_recorded_slate"],
    "metric_definitions": {
        "known_click_recall@100": "PRIMARY. Per request with known clicks: clicked IDs in the "
        "exported top-100 (score desc, article ID asc) / all its original clicked IDs; clicks "
        "rejected by eligibility and empty pools count against it; macro mean over requests.",
        "observed_click_ndcg@100": "SECONDARY. DCG@100 with gain 1 for an original known click, "
        "0 observed gain otherwise (bookkeeping, not a negative label), over the exported order; "
        "ideal DCG places all original known clicks first, capped at 100, including excluded "
        "or missed clicks; macro mean over the same requests.",
        "eligible_only_recall@100": "Recall over eligible clicks only (reported separately).",
        "ties_at_cutoff": "Requests where an article outside the top 100 had the same score "
        "as the 100th; the article-ID rule decided membership.",
        "diagnostic mean_impression_auc": D.METRIC_DEFINITIONS["mean_impression_auc"]
        + " DIAGNOSTIC only; not used for selection.",
        "recorded-slate ndcg@5": "Diagnostic only; nDCG@5 is the Stage 2 final-list metric."},
    "selection": {**{k: v for k, v in SELECTION.items() if k != "pool_policy"},
                  **({} if TRAIN_MODEL else {"objective": SELECTION_PROVENANCE["selected_by"],
                                             "evaluation_declaration_objective": SELECTION_OBJECTIVE}),
                  "weights_selected_under": WEIGHTS_SELECTED_UNDER,
                  "selected_epoch": BEST["epoch"], "selected_key": BEST.get("key"),
                  "selected_metrics": {k: v for k, v in (BEST.get("metrics") or {}).items()
                                       if not str(k).startswith("_")},
                  "provenance": SELECTION_PROVENANCE},
    "comparisons": {"table": COMPARISON_TABLE.to_dict("records"), "bootstrap": COMPARISONS,
                    "note": "single seed; paired user-cluster bootstrap of Recall@100 over the "
                            "selection sample (identical requests and full pools); AUC diagnostic"},
    "diagnostics": DIAGNOSTICS, "attention": ATTENTION_DIAG,
    "catalogue": {"train_in_sample": (CATALOGUE_RECORDS.get("train") or {}).get("metrics"),
                  "test": {k: v for k, v in ((CATALOGUE_RECORDS.get("test") or {}).get("metrics")
                                             or {}).items() if "recall" not in k} or None,
                  "skipped_this_session": CATALOGUE_SKIPPED},
    "test": "unlabelled; no metric reported",
    "request_scope": REQUEST_SCOPE, "mini_sample": IS_MINI_SAMPLE,
    "completion_scope": ("complete" if (not any(LIMITED.values()) and TRAIN_REQUESTS is None
                                        and not CATALOGUE_SKIPPED
                                        and all(s in CATALOGUE_RECORDS for s in D.SPLITS))
                         else "sampled development" if (any(LIMITED.values()) or TRAIN_REQUESTS)
                         else "incomplete (catalogue splits missing)"),
}
D.write_json(METRICS, RUN_DIR / "metrics.json")

EXPORT_MANIFEST = {
    "export_schema_version": D.EXPORT_SCHEMA_VERSION, "prediction_pool": "time_eligible_catalogue",
    "eligibility_policy": POLICY.to_json(), "shortlist_size": SHORTLIST_SIZE,
    "final_display_size": FINAL_DISPLAY_SIZE,
    "catalogue_sources": {k: {"path": str(PATHS[f"articles_{k}"]), "articles": int(len(v))}
                          for k, v in SNAPSHOT_IDS.items()},
    "union_catalogue_articles": int(len(UNION_IDS)),
    "same_pool_for_all_splits": True,
    "request_scope": REQUEST_SCOPE, "protocol_sha256": SELECTION["protocol_sha256"],
    "splits": {s: {"rankings": r["files"]["rankings"],
                   "eligibility_manifest": r["files"]["manifest"],
                   "catalogue_scores": r["files"]["scores"],
                   "catalogue_scores_complete": r["full_score_export"],
                   "score_requests_exported": r["score_requests_exported"],
                   "sampled_requests": r["sampled_requests"],
                   "requests_scored": r["requests_scored"],
                   "requests_in_split": r["requests_in_split"],
                   "recorded_slate_diagnostics": str(DIAG_PATHS[s].relative_to(RUN_DIR)),
                   "request_ids_file": r.get("request_ids_file"),
                   "requests": r["metrics"]["requests"],
                   "eligible_pairs": r["validation"]["eligible_pairs"],
                   "empty_pools": r["validation"]["empty_pools"]}
               for s, r in CATALOGUE_RECORDS.items()},
    "not_scored_splits": [s for s in D.SPLITS if s not in CATALOGUE_RECORDS],
    "score_semantics": {"score": "logit (fp32)", "click_probability": "sigmoid(score); uncalibrated "
                        "BCE estimate conditional on logged slates; need not sum to 1"},
    "unshown_outcomes": "unknown (not negative labels)",
}
D.write_json(EXPORT_MANIFEST, RUN_DIR / "export_manifest.json")

SOURCE_RUN_CONFIG = (json.loads((PERSIST_ROOT / LOAD_RUN_ID / "run_config.json").read_text())
                     if not TRAIN_MODEL else None)
RUN_CONFIG = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED,
    "created": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "mode": "train" if TRAIN_MODEL else "load",
    "source_run_id": None if TRAIN_MODEL else LOAD_RUN_ID, "smoke_test": SMOKE_TEST,
    "mini_sample": IS_MINI_SAMPLE, "request_scope": REQUEST_SCOPE, "protocol": PROTOCOL,
    "completion_scope": METRICS["completion_scope"],
    "text_encoder": TEXT_ENCODER,
    "versions": VERSIONS, "code": GIT_INFO,
    "data_root": str(DATA_ROOT), "inputs": D.describe_inputs(PATHS, HASH_INPUTS),
    "path_notes": PATH_NOTES, "feature_case": FEATURE_CASE,
    "feature_table_columns_used": D.BASE_FEATURE_COLUMNS + DEVICE_COLUMNS,
    "engineered_aggregates_used": False,
    "input_tables": TABLE_INFO, "catalogue": CATALOGUE_INFO, "article_table": ARTICLE_INFO,
    "vectors": VECTOR_INFO, "compatible_vector_copies": VECTOR_COPIES or None,
    "candidate_age_check": AGE_CHECK,
    "eligibility": {"policy": POLICY.to_json(), "summary": ELIGIBILITY,
                    "known_click_eligibility": {s: v["summary"] for s, v in CLICK_ELIGIBILITY.items()
                                                if v.get("labelled")},
                    "union_catalogue_articles": int(len(UNION_IDS)),
                    "excluded_unknown_publication": union_index.excluded_missing_published},
    "catalogue_feasibility": FEASIBILITY, "catalogue_records": CATALOGUE_RECORDS,
    "export_manifest": "export_manifest.json",
    "output_root": str(OUTPUT_ROOT), "resume_check": RESUME_CHECK,
    "history": {s: HISTORIES[s].stats for s in D.SPLITS},
    "history_policy": {
        "source": "each split's fixed pre-period history.parquet; no in-period behaviour appended",
        "cutoff": "every retained event strictly before the split's earliest impression (asserted)",
        "order": "chronological; original list position breaks timestamp ties; repeat reads kept",
        "truncation": f"most recent {HISTORY_MAX_LEN} events, oldest first, right-padded with masks",
        "empty_history": "learned empty-context vector + history_empty flag",
        "unknown_articles": "kept with missing/unknown category/type indices and in_catalogue=0",
        "missing_vectors": "zero placeholder input + has_text/has_image flag; rows kept; "
                           "training hides text for MODEL_HPARAMS['text_dropout'] of candidates",
    },
    "features": {"context": ARRAYS["ordered"]["train"].context_names,
                 "candidate": ARRAYS["ordered"]["train"].candidate_names,
                 "event": ARRAYS["ordered"]["train"].event_names,
                 "article_numeric": ARTICLES.numeric_names,
                 "removed_pool_dependent": ["recorded slate_size", "within-slate age rank"],
                 "excluded_for_leakage": ["behaviours read_time/scroll_percentage",
                                          "next_read_time/next_scroll_percentage",
                                          "total_inviews/total_pageviews/total_read_time"]},
    "preprocessing": {"fitted_on": "train" + ("" if TRAIN_REQUESTS is None else
                                              f" ({TRAIN_REQUESTS} hash-sampled requests)"),
                      "train_candidate_rows": int(len(FRAMES["train"])),
                      "file": "model/preprocessing.json"},
    "model": {"config": MAIN_CONFIG.to_json(), **M.count_parameters(model),
              "frozen": ["text vectors (gemini-embedding-2, 768-d)", "image vectors (EB-NeRD, 1024-d)"],
              "trained": ["all embeddings, projections, LSTM, attention, head"]},
    "training": SOURCE_RUN_CONFIG["training"] if not TRAIN_MODEL else {
        "config": dict(TRAIN_HPARAMS), "loss": "BCEWithLogitsLoss over recorded candidates",
        "optimizer": "AdamW", "scheduler": f"ReduceLROnPlateau(max {SELECTION_OBJECTIVE})",
        "selected_epoch": BEST["epoch"], "seconds": TRAIN_SECONDS},
    "selection": {**SELECTION, "lr_schedule": "ReduceLROnPlateau on the primary objective only"},
    "weights_selected_under": WEIGHTS_SELECTED_UNDER,
    "selection_provenance": SELECTION_PROVENANCE,
    "controls_trained_in_this_run": {v: {"best_epoch": r["best_epoch"]} for v, r in ABLATIONS.items()},
    "tuning": TUNING_SUMMARY,
    "export_schema_version": D.EXPORT_SCHEMA_VERSION, "prediction_pool": "time_eligible_catalogue",
    "shortlist_size": SHORTLIST_SIZE, "final_display_size": FINAL_DISPLAY_SIZE,
    "recorded_slate_diagnostics": DIAG_INFO,
    "export_requests_limit": EXPORT_REQUESTS,
    "score": {"score_type": "logit", "source": "selected checkpoint, fp32 inference",
              "click_probability": "sigmoid(score) from BCE training; uncalibrated, conditional "
              "on logged slates; need not sum to 1" if EXPORT_CLICK_PROBABILITY else None,
              "train_predictions": "in-sample (not out-of-fold)"},
    "persistent_location": str(PERSIST_RUN_DIR) if PERSIST_TO_DRIVE else None,
}
D.write_json(RUN_CONFIG, RUN_DIR / "run_config.json")
log("Run bundle written to", RUN_DIR)

In [ ]:
# Persist the complete run folder to Drive and verify sizes + hashes.
PERSIST_RESULT = None
if PERSIST_TO_DRIVE:
    D.write_json(D.file_manifest(RUN_DIR), RUN_DIR / "manifest.json")
    PERSIST_RESULT = D.persist_directory(RUN_DIR, PERSIST_RUN_DIR)
    log("Persisted:", {k: v for k, v in PERSIST_RESULT.items() if k != "manifest"})
    assert PERSIST_RESULT["verified"], "Drive copy differs from the local run folder"
else:
    log("PERSIST_TO_DRIVE=False: the run exists only on this runtime; persistence incomplete.")

## 8. Reload check

This check uses only the **persistent copy**. It:
1. imports the bundled code;
2. rebuilds the model from `model_config.json` and loads `model_state.pt` with
   `weights_only=True` (no retraining);
3. rebuilds the inputs with the bundled `preprocessing.json` and
   `catalogue_policy.json` (no refitting);
4. predicts a fixed sample of validation and test recorded slates and compares
   them with the diagnostic CSVs read back from Drive;
5. recomputes catalogue shortlists for sampled requests and requires them to equal
   the exported top-100 lists.

The result is written to `reload_check.json`. It counts as evidence only after
this cell has actually run and printed `passed: True`.

In [ ]:
source_dir = PERSIST_RUN_DIR if PERSIST_TO_DRIVE else RUN_DIR
bundle = source_dir / "model"
spec_m = importlib.util.spec_from_file_location("bundled_lstm_model", bundle / "code" / "lstm_model.py")
spec_d = importlib.util.spec_from_file_location("bundled_lstm_data", bundle / "code" / "lstm_data.py")
BM, BD = importlib.util.module_from_spec(spec_m), importlib.util.module_from_spec(spec_d)
sys.modules[spec_m.name], sys.modules[spec_d.name] = BM, BD   # needed by @dataclass
spec_m.loader.exec_module(BM); spec_d.loader.exec_module(BD)

prep = json.loads((bundle / "preprocessing.json").read_text())
cfg = BM.ModelConfig.from_json(json.loads((bundle / "model_config.json").read_text()))
maps = np.load(bundle / "mappings.npz")
id_vocab = dict(zip(maps["id_vocab_ids"].tolist(), maps["id_vocab_rows"].tolist()))
reload_articles, _ = BD.build_article_table(
    CATALOGUE, all_ids, prep["feature_vocabulary"], prep["article_stats"], TEXT_LOOKUP,
    IMAGE_LOOKUP, prep["text_dim"], prep["image_dim"], prep["max_topics"], id_vocab)
reloaded = BM.build_model(cfg, BM.article_table_tensors(reload_articles, DEVICE), DEVICE)
reloaded.load_state_dict(torch.load(bundle / "model_state.pt", map_location=DEVICE, weights_only=True))
reloaded.eval()

RELOAD = {"source": str(bundle), "tolerance_abs_logit": RELOAD_TOLERANCE, "splits": {},
          "article_table_matches_saved": bool(np.array_equal(maps["article_table_ids"],
                                                             reload_articles.article_ids))}
for split in ("validation", "test"):
    arrays = BD.build_split_arrays(split, FRAMES[split], OFFSETS[split], HISTORIES[split],
                                   reload_articles, prep["numeric_stats"], prep["device_columns"])
    if cfg.variant == "attention_shuffled":
        arrays = BD.permute_history(arrays, prep["shuffle_seed"] + BD.SPLITS.index(split))
    subset = np.sort(D.first_by_priority(REQUESTS[split], EXPORT_BIDS[split], split, SAMPLING_SEED,
                                         min(RELOAD_CHECK_IMPRESSIONS, len(REQUESTS[split]))))
    logits = BM.predict_logits(reloaded, BM.SplitTensors(arrays, DEVICE), impression_subset=subset)
    fresh = BD.score_frame(arrays, logits)
    fresh = fresh[fresh["score"].notna()]
    exported = pd.read_csv(source_dir / DIAG_PATHS[split].relative_to(RUN_DIR),
                           dtype={"behaviour_id": str, "article_id": "int64"},
                    float_precision="round_trip")
    merged = fresh.merge(exported, on=["behaviour_id", "article_id"],
                                           suffixes=("_reload", "_export"), validate="one_to_one")
    diff = np.abs(merged["score_reload"] - merged["score_export"]).to_numpy()
    RELOAD["splits"][split] = {"impressions": int(len(subset)), "candidates": int(len(merged)),
                               "max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean())}
    # Catalogue path: recompute shortlists for sampled requests from the reloaded
    # model and compare with the exported top-100 files (read back from the copy).
    if split in CATALOGUE_RECORDS:
        policy_saved = json.loads((bundle / "catalogue_policy.json").read_text())
        assert policy_saved["policy"]["name"] == POLICY.name, "saved pool policy differs"
        index = BD.build_catalogue_index(split, "union", UNION_IDS, reload_articles)
        picks = D.first_by_priority(REQUESTS[split], EXPORT_BIDS[split], split, SAMPLING_SEED,
                                    min(CATALOGUE_RELOAD_REQUESTS, len(REQUESTS[split])))
        scorer = BM.CatalogueScorer(reloaded, BM.SplitTensors(arrays, DEVICE), index,
                                    prep["numeric_stats"]["log_candidate_age_h"],
                                    None, CATALOGUE_CHUNK, requests=picks)
        k = policy_saved["shortlist_size"]
        ranks = pd.read_csv(source_dir / CATALOGUE_RECORDS[split]["files"]["rankings"],
                            dtype={"behaviour_id": str}).set_index("behaviour_id")["article_ids"]
        bids = BD.behaviour_ids(arrays.user_id, arrays.impression_id)
        identical = consistent = 0
        for i in picks:
            top_ids, top_logits, ids, logits = scorer.shortlist(int(i), k)
            mine, exported_list = top_ids.cpu().tolist(), json.loads(ranks[bids[i]])
            if mine == exported_list:
                identical += 1; consistent += 1
                continue
            # Allow only reorderings/swaps among scores within the tolerance of each
            # other or of the cut-off (GPU nondeterminism at near-ties).
            score = dict(zip(ids.cpu().tolist(), logits.cpu().tolist()))
            cut = float(top_logits[-1]) if len(top_logits) else 0.0
            swapped = set(mine) ^ set(exported_list)
            ordered = all(score[a] >= score[b] - RELOAD_TOLERANCE
                          for a, b in zip(exported_list, exported_list[1:]))
            consistent += int(ordered and len(exported_list) == len(mine)
                              and all(abs(score[a] - cut) <= RELOAD_TOLERANCE for a in swapped))
        RELOAD["splits"][split]["catalogue_requests_checked"] = int(len(picks))
        RELOAD["splits"][split]["catalogue_shortlists_identical"] = int(identical)
        RELOAD["splits"][split]["catalogue_shortlists_consistent_within_tolerance"] = int(consistent)
RELOAD["passed"] = RELOAD["article_table_matches_saved"] and all(
    v["max_abs_diff"] <= RELOAD_TOLERANCE and v["candidates"] > 0
    and v.get("catalogue_shortlists_consistent_within_tolerance", 0) == v.get("catalogue_requests_checked", 0)
    for v in RELOAD["splits"].values())
D.write_json(RELOAD, RUN_DIR / "reload_check.json")
if PERSIST_TO_DRIVE:
    shutil.copy2(RUN_DIR / "reload_check.json", PERSIST_RUN_DIR / "reload_check.json")
log("Reload check:", RELOAD)
assert RELOAD["passed"], "reloaded predictions differ from exported scores beyond tolerance"

## 9. Business interpretation and handoff

**Fill in after the run, from Sections 6–8.**
- **Validation finding (development data):** catalogue known-click Recall@100 …
  (primary; full validation catalogue), observed-click nDCG@100 … (secondary);
  selected epoch … by Recall@100 on the fixed selection sample; eligible-only
  recall …, eligibility coverage …; diagnostic recorded-slate AUC ….
- **Business link:** Stage 1 protects personal relevance through shortlist
  membership; Stage 2 applies continuity, freshness and diversity within the
  top 100. The candidate-specific attention shows *which* past reading drives each
  recommendation, which helps the reranking owner explain topical concentration.
  Read-time and scroll inputs represent depth of past engagement. That does not
  establish continued reading.
- **Limits:**
  - These are offline click metrics on logged slates. Catalogue recall measures
    recovery of observed clicks only; unshown articles' outcomes are unknown.
  - Most test-catalogue articles have no text vector (Section 4 coverage); their
    scores rely on metadata, image vectors and text-dropout training.
  - Validation was used for selection.
  - Single seed.
  - Histories are fixed pre-period snapshots: users' in-week reads are not used.
  - Test labels are unavailable.
  - Probabilities are uncalibrated.
- **For the reranking/evaluation owner:**
  - `rankings/{split}/submission_top_100.csv` (Stage 1 shortlists, every request).
  - Catalogue scores: `predictions/{split}/submission_click_probabilities.csv` when
    complete, otherwise the declared request sample under `sampled_requests/`.
    `export_manifest.json` states which (export schema v3).
  - `diagnostics/recorded_slates/{split}_scores.csv` for the labelled benchmark.
  - Join labels with `(behaviour_id, article_id)`.
  - Reload instructions: set `TRAIN_MODEL = False` and `LOAD_RUN_ID` to this run.

In [ ]:
# Small files to return for review (no datasets or model binaries).
RETURN_FILES = ["metrics.json", "run_config.json", "training_history.csv", "reload_check.json",
                "learning_curves.png", "selection/validation_requests.csv",
                "resume_check.json", "catalogue_feasibility.json", "export_manifest.json",
                "requirements.txt", "logs/notebook_log.txt", "attention_recency.png",
                "manifest.json"]
RETURN_FILES += [f"ablations/{v}/summary.json" for v in ABLATIONS]
RETURN_FILES += [f"catalogue/{s}.json" for s in CATALOGUE_RECORDS]
review_zip = RUN_DIR / f"review_{RUN_ID}.zip"
import zipfile
with zipfile.ZipFile(review_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in RETURN_FILES:
        path = RUN_DIR / name
        if path.is_file():
            zf.write(path, name)
    if TUNING_SUMMARY:
        trials = Path(TUNING_SUMMARY["trials_csv"])
        if trials.is_file():
            zf.write(trials, "tuning/trials.csv")
if PERSIST_TO_DRIVE:
    shutil.copy2(review_zip, PERSIST_RUN_DIR / review_zip.name)
    # Refresh the manifest so it also covers reload_check.json and the review zip.
    D.write_json(D.file_manifest(RUN_DIR), RUN_DIR / "manifest.json")
    final = D.persist_directory(RUN_DIR, PERSIST_RUN_DIR)
    log("Final persistence verified:", final["verified"], "| files:", final["files"])
print("Persistent run folder:", PERSIST_RUN_DIR if PERSIST_TO_DRIVE else "(not persisted)")
print("Review bundle to return with the executed notebook:",
      (PERSIST_RUN_DIR if PERSIST_TO_DRIVE else RUN_DIR) / review_zip.name)
print(COMPARISON_TABLE.to_string(index=False))